# Educational Data

## Overview

Detailed information on Connecticut School Districts is combined from multiple sources. This is necessary for a holistic view of performance and resources. Much of the work is done in conforming across sources so that it may be combined with other information about the towns, such as their socioeconomic peer groups and financing sources.

The data suffers from two disruptions. First is the change in methodology and reporting in 2017, in which functional expenditures are grouped differently. Much work has been done to match and impute to develop continuity over time. The second is the Covid pandemic, which interrupted data collection and reporting for two years. Those years are simply missing.

## Data Sources

SocioEconomic Data from National Center for Education Statistics https://nces.ed.gov/programs/edge/Home  
Inflation is from Bureau of Economic Analysis. Click on interactive tables https://www.bea.gov/data/prices-inflation/gdp-price-deflator 

**CT DOE EdSight for most district-level data:**
- staffing: https://public-edsight.ct.gov/educators/fte-staffing
- attrition (not as recent): https://public-edsight.ct.gov/educators/attrition?language=en_US
- performance (accountability index): https://public-edsight.ct.gov/overview/next-generation-accountability-dashboard?language=en_US
- expenditures by function: https://public-edsight.ct.gov/overview/per-pupil-expenditures-by-function---district?language=en_US
- spending for 2016 & earlier from "related links" [per pupil](https://public-edsight.ct.gov/overview/per-pupil-expenditures-by-function---district/per-pupil-expenditures-2016-17-and-earlier?language=en_US) and [annual totals](https://public-edsight.ct.gov/overview/per-pupil-expenditures-by-function---district/total-annual-expenditures-by-type-2016-17-and-earlier?language=en_US)
- enrollment is from expenditure files or https://public-edsight.ct.gov/students/enrollment-dashboard/public-school-enrollment-export?language=en_US
See separate notebook for socioeconomic classification.
- Based on district-level demographic data from https://nces.ed.gov/programs/edge/Demographic/ACS

In [1]:
from os import path
import glob
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
def get_type(x):

    m = re.match(r"(.*)( School District$)", x)
    
    if x.strip().startswith('Regional'):
        return 'Regional'
    elif x.strip().endswith('Academy District'):
        return 'Academy'
    elif x.strip().startswith('Unified'):
        return 'Unified' 
    elif re.search("Charter", x):
        return 'Charter'
    elif x.startswith(('Elm City', 'Odyssey', 'Capital Prep', 'Common Ground')):
        return 'Charter'
    elif m:
        return 'Town'
    else:
        return "Other"

In [3]:
def get_town(x):

    m = re.match(r"(.*)( School District$)", x)
    
    if x.strip().startswith('Regional'):
        return 'Regional'
    elif x.strip().endswith('Academy District'):
        return 'Academy'
    elif x.strip().startswith('Unified'):
        return 'Unified' 
    elif re.search("Charter", x):
        return 'Charter'
    elif x.startswith(('Elm City', 'Odyssey', 'Capital Prep', 'Common Ground')):
        return 'Charter'
    elif m:
        return m.group(1).strip()
    else:
        return "Other"

## Import data

### Inflation
Uses the December / Q4 numbers because 
- they are halfway through the school year
- the year column aligns with the starting year of the school year

In [4]:
ipd = pd.read_csv('/Users/joel/Desktop/Granby Benchmarking/price_index_ipd_clean.csv')
ipd = ipd[ipd['Quarter'] == 'Q4'].drop(columns=['Quarter'])
ipd.columns = [c.strip() for c in ipd.columns]
ipd['School Year'] = ipd['Year'].astype('string')
ipd.drop(columns=['Year'], inplace=True)
ipd['IPD'] = ipd['IPD']/100

pce = pd.read_csv('/Users/joel/Desktop/Granby Benchmarking/price_index_pce_clean.csv')[['Year', 'Month', 'PCE']]
pce = pce[pce['Month'] == 'DEC'].drop(columns=['Month'])
pce.columns = [c.strip() for c in pce.columns]
pce['School Year'] = pce['Year'].astype('string')
pce.drop(columns=['Year'], inplace=True)
pce['PCE'] = pce['PCE']/100

### Students

In [5]:
enrollment = pd.read_excel('/Users/joel/Desktop/Granby Benchmarking/education/dd1174.xlsx')
enrollment_swd = pd.read_excel('/Users/joel/Desktop/Granby Benchmarking/education/dd1174 swd.xlsx')
enrollment = pd.concat([enrollment, enrollment_swd[enrollment_swd['Student Group'].notna()]])

enrollment.columns = [c.replace('*','') for c in enrollment.columns]
enrollment['Student Group'] = enrollment['Student Group'].fillna('Total').str.strip()
enrollment = pd.concat([enrollment.drop(columns=['District']), enrollment['District'].str.extract(r"(?P<District>.*)\((?P<District_Code>\d*)\)")], axis=1)
enrollment['School Year'] = enrollment['School Year'].str.strip()
enrollment['District'] = enrollment['District'].str.strip()

#drop district code - inconsistent
enrollment.drop(columns=['District_Code'], inplace=True)

enrollment.head(5)

/Users/joel/opt/anaconda3/envs/base312/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/joel/opt/anaconda3/envs/base312/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Student Group,School,School Year,Student Count,District
0,Total,District-level,2010-11,932.0,Woodstock School District
1,Total,District-level,2011-12,894.0,Woodstock School District
2,Total,District-level,2012-13,897.0,Woodstock School District
3,Total,District-level,2013-14,904.0,Woodstock School District
4,Total,District-level,2014-15,873.0,Woodstock School District


In [6]:
enrollment['Student Group'].value_counts()

Student Group
Total                            3204
High Needs                       3204
Non-High Needs                   3204
Students with Disabilities       3204
Students without Disabilities    3204
Name: count, dtype: int64

### Teacher attrition

Key for retention is that they stayed in the district, whether in the classroom or not.

In [7]:
# This contains only 
# - district total ('school/program')
# - classroom teachers ('staff type')
attrition = pd.read_excel('/Users/joel/Desktop/Granby Benchmarking/education/Educator Attrition.xlsx')\
              .rename(columns={'School Year':'Year', 'Student Count*': 'Student Count'})

attrition['Town'] = attrition['District'].apply(lambda x: get_town(x))
attrition['Type'] = attrition['District'].apply(lambda x: get_type(x))
attrition['District Code'] = attrition['District Code'].astype('string').str.zfill(7).str.slice(0,3)
attrition['Code'] = attrition['District Code'].astype('int')
attrition['Retained'] = attrition['Rate'].where(attrition['Turnover Description'].str.startswith('Stayed'), 0)
attrition['Elsewhere'] = attrition['Rate'].where(attrition['Turnover Description'].str.startswith('Left the district, stayed in the classroom'), 0)

#drop district code - inconsistent
attrition.drop(columns=['District Code'], inplace=True)

attrition.sort_values(by=['District', 'Year'], inplace=True)

/Users/joel/opt/anaconda3/envs/base312/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [8]:
attrition.columns

Index(['Year', 'District', 'School/Program', 'Low Grade', 'High Grade',
       'Turnover (Yes/No)', 'Turnover Description', 'Count', 'Total', 'Rate',
       'Student Count', 'Staff Type', 'SchoolCode', 'Organization Type',
       'Alliance', 'CommishNetwork', 'Focus', 'Opportunity', 'Priority',
       'Turnaround', 'Town', 'Type', 'Code', 'Retained', 'Elsewhere'],
      dtype='object')

In [9]:
attrition[['District', 'Organization Type', 'Year', 'Turnover Description',
           'Count', 'Retained', 'Elsewhere', 'Rate', 'Student Count']].head(1800).tail(15)

,District,Organization Type,Year,Turnover Description,Count,Retained,Elsewhere,Rate,Student Count
1779,Granby School District,Public School Districts,2021-22,"Left the district, stayed in the classroom",5,0.000000,0.034247,0.034247,1765
1780,Granby School District,Public School Districts,2021-22,"Left the district, left the classroom",0,0.000000,0.000000,0.000000,1765
1781,Granby School District,Public School Districts,2021-22,Left CT public schools,10,0.000000,0.000000,0.068493,1765
1782,Granby School District,Public School Districts,2021-22,"Stayed in the district, stayed in the classroom",131,0.897260,0.000000,0.897260,1765
1797,Granby School District,Public School Districts,2021-22,"Stayed in the district, left the classroom",0,0.000000,0.000000,0.000000,1765
1774,Granby School District,Public School Districts,2022-23,"Left the district, left the classroom",1,0.000000,0.000000,0.006757,1738
1778,Granby School District,Public School Districts,2022-23,"Left the district, stayed in the classroom",7,0.000000,0.047297,0.047297,1738
1789,Granby School District,Public School Districts,2022-23,"Stayed in the district, left the classroom",5,0.033784,0.000000,0.033784,1738
1798,Granby School District,Public School Districts,2022-23,Left CT public schools,14,0.000000,0.000000,0.094595,1738
1799,Granby School District,Public School Districts,2022-23,"Stayed in the district, stayed in the classroom",121,0.817568,0.000000,0.817568,1738


In [10]:
attrition[attrition['Turnover Description'] == "Stayed in the district, stayed in the classroom"]['Year'].value_counts()

Year
2018-19    191
2019-20    190
2020-21    190
2021-22    190
2022-23    190
2023-24    188
Name: count, dtype: int64

#### Usable columns for teacher attrition

Dataframe **retention**

In [11]:
retention = attrition.groupby(['District', 'Year'])\
         .agg({'Count': 'sum', 'Retained': 'sum', 'Elsewhere': 'sum', 'Student Count':'max'})\
        .reset_index()\
        .rename(columns={'Count': 'Classroom Teachers', 'Retained': 'Retention Rate', 'Elsewhere': 'Redistrict Rate'})

retention['District'] = retention['District'].str.strip()
retention['Year'] = retention['Year'].str.strip()

retention.head()

,District,Year,Classroom Teachers,Retention Rate,Redistrict Rate,Student Count
0,Achievement First Bridgeport Academy District,2018-19,65,0.692308,0.153846,1093
1,Achievement First Bridgeport Academy District,2019-20,66,0.681818,0.106061,1110
2,Achievement First Bridgeport Academy District,2020-21,65,0.692308,0.200000,1127
3,Achievement First Bridgeport Academy District,2021-22,66,0.545455,0.227273,1105
4,Achievement First Bridgeport Academy District,2022-23,50,0.600000,0.140000,1084


### Accountability Index
Wide-ranging performance measures that include test scores, post-school preparedness, other enrichment, and so on.

#### District-level accountability file
Has reasonably named columns for the success indicators.

In [12]:
# raw numbers for all students
#pd.read_excel('/Users/joel/Desktop/Granby Benchmarking/education/dd7582.xlsx')

# % of points
accountability = pd.read_excel('/Users/joel/Desktop/Granby Benchmarking/education/dd7070.xlsx')
newcols = [c.replace(" (% of points earned)", "").strip() for c in accountability.columns]
newcols = [c.replace(" - All Students", "").strip() for c in newcols]
accountability.columns = newcols
accountability.rename(columns={'AccountabilityIndex': 'Accountability Index'}, inplace=True)
accountability['Town'] = accountability['District'].apply(lambda x: get_town(x))
accountability['Type'] = accountability['District'].apply(lambda x: get_type(x))
accountability['Granby'] = accountability['Town'].where(accountability['Town'] == 'Granby', 'Other')

/Users/joel/opt/anaconda3/envs/base312/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [13]:
accountability.Year.value_counts().sort_index()

Year
2014-15    189
2015-16    192
2016-17    192
2017-18    192
2018-19    192
2021-22    190
2022-23    190
2023-24    190
2024-25    189
Name: count, dtype: int64

In [14]:
accountability.columns

Index(['Year', 'Accountability Index', 'District', 'School', 'ELA PI',
       'ELA PI - High Needs', 'Math PI', 'Math PI - High Needs', 'Science PI',
       'Science PI - High Needs', 'ELA Growth', 'ELA Growth - High Needs',
       'Math Growth', 'Math Growth - High Needs',
       'Progress Toward English Proficiency - Literacy',
       'Progress Toward English Proficiency - Oral', 'Chronic Absenteeism',
       'Chronic Absenteeism - High Needs', 'CCR - Taking Courses',
       'CCR - Passing Exams', 'On-track to HS Graduation', '4-yr Graduation',
       '6-yr Graduation - High Needs', 'Postsecondary Entrance',
       'Physical Fitness', 'Arts Access', 'Town', 'Type', 'Granby'],
      dtype='object')

### School level
#### long for plotting

In [15]:
# mapping to understandable
_meta = pd.read_excel('/Users/joel/Desktop/Granby Benchmarking/education/nextgenmetadata.xlsx')[['Column Name', 'Description']].set_index('Column Name')
ind_meta = {}
for r, s in _meta.iterrows():
    ind_meta[r] = s.item()


In [16]:
# read indicators
acctfiles = '/Users/joel/Desktop/Granby Benchmarking/education/nextgenacct*.csv'
flist = sorted(glob.glob(acctfiles))

dfs = []
for f in flist:
    dd = pd.read_csv(f, header=0, nrows=1, encoding='cp1252')
    yr = dd.columns[0][-8:]
    _df = pd.read_csv(f, skiprows=1, header=0, encoding='cp1252')
    _df['Year'] = yr.strip()
    dfs.append(_df)

accountible = pd.concat(dfs)
_indicators = [x for x in accountible.columns if (x.startswith('Ind')) & (x.endswith('Rate'))]

In [17]:
# get var name, then convert to description
indicator_map = {}
for k in _indicators:
    x = ind_meta[k]
    dash = x.find(" - ") if x.find(" - ") > 0 else 999
    colon = x.find(":") if x.find(":") > 0 else 999
    paren = x.find(" (") if x.find(" (") > 0 else 999
    liter = x.find('Literacy Rate')
    oral = x.find('Oral Rate')
    cte = x.find('CTE')
    cw = x.find('Coursework')
    ap = x.find('Passing Exams')

    x = x[0:min(dash, colon, paren)].strip()
    x = x + ' (Oral)' if oral > 0 else x
    x = x + ' (Literacy)' if liter > 0 else x
    x = x + ' (Courses)' if ((cte > 0) | (cw > 0))else x
    x = x + ' (Exams)' if ap > 0 else x
    x = x.replace("Language Proficiency ","")
    x = x.replace(" Points Possible","")
    x = x.replace("Assessment ","")
    x = x.replace("High School","HS")
    x = x.strip()
    indicator_map[k] = x

needs_map = {}
for k in _indicators:
    x = ind_meta[k]
    j = 'High Needs' if k.find('HN') > 0 else 'All'
    j = 'Not High Needs' if k.find('NHN') > 0 else j
    needs_map[k] = j

In [18]:
indicator_map

{'Ind1ELA_All_Rate': 'ELA Performance Index',
 'Ind1ELA_HN_Rate': 'ELA Performance Index',
 'Ind1ELA_NHN_Rate': 'ELA Performance Index',
 'Ind1Math_All_Rate': 'Math Performance Index',
 'Ind1Math_NHN_Rate': 'Math Performance Index',
 'Ind1Math_HN_Rate': 'Math Performance Index',
 'Ind1Sci_All_Rate': 'Science Performance Index',
 'Ind1Sci_HN_Rate': 'Science Performance Index',
 'Ind1Sci_NHN_Rate': 'Science Performance Index',
 'Ind2ELA_All_Rate': 'ELA Academic Growth',
 'Ind2Math_All_Rate': 'Math Academic Growth',
 'Ind2ELA_HN_Rate': 'ELA Academic Growth',
 'Ind2ELA_NHN_Rate': 'ELA Academic Growth',
 'Ind2Math_HN_Rate': 'Math Academic Growth',
 'Ind2Math_NHN_Rate': 'Math Academic Growth',
 'Ind2LEP_LTCY_Rate': 'Progress Toward English (Literacy)',
 'Ind2LEP_ORAL_Rate': 'Progress Toward English (Oral)',
 'Ind3ELA_All_Rate': 'Participation Rate ELA',
 'Ind3ELA_HN_Rate': 'Participation Rate ELA',
 'Ind3ELA_NHN_Rate': 'Participation Rate ELA',
 'Ind3Math_All_Rate': 'Participation Rate Math'

In [19]:
#source: https://edsight.ct.gov/relatedreports/Next%20Gen%20Detailed%20Presentation_Oct%202025.pdf
# https://edsight.ct.gov/relatedreports/Using_Accountability_Results_to_Guide_Improvement.pdf
def get_divisor(k):
    """ divisor for the score """
    if k.find('Graduation') >= 0:
        return .94
    elif k.find('Arts') >= 0:
        return .60
    elif ((k.find('Growth') >= 0) |
          (k.find('Participation') >= 0) |
          (k.find('Participation') >= 0) |
          (k.find('Physical') >= 0)
         ):
        return 1
    elif k.find('Absentee') >= 0:
        # need to correct with (30 - x) / .25
        return 1 
    else:
        return .75

def fitness_mult(x):
    """ adjust phys fitness using Ind11ParticipationRate """
    if x < 50:
        return 0
    elif x < 70:
        return .25 * 50
    elif x < 90:
        return .5 * 50
    else:
        return 1

In [20]:
accountible.rename(inplace=True, columns={'RptngDistrictName': 'District', 'SchoolName':'School Name'})
accountible['School Year'] = accountible['Year'].str.slice(0,4)

accountible['District Code'] = accountible['ReportingDistrictCode'].apply(lambda x: re.search(r'.*"(\d+)"',x).group(1))
accountible['School Code'] = accountible['SchoolCode'].apply(lambda x: re.search(r'.*"(\d+)"',x).group(1))
accountible['Code'] = accountible['District Code'].str.slice(0,3).astype('int')
accountible['Low Grade'] = accountible['SchoolLowGrade'].str.replace('PK', '0').str.replace('K', '0').str.replace('District', '-1').astype('int')
accountible['High Grade'] = accountible['SchoolHighGrade'].str.replace('PK', '0').str.replace('K', '0').str.replace('District', '-1').astype('int')
accountible['Category'] = accountible['Category'].str.replace("Tot","")


In [21]:
accountible['Ind11ParticipationRate'] = accountible['Ind11ParticipationRate'].where(accountible['Ind11ParticipationRate'] != ".", np.nan).astype('float')
accountible['Ind11FitnessRate'] = accountible['Ind11FitnessRate'].where(accountible['Ind11FitnessRate'] != ".", np.nan).astype('float')
accountible['Ind11FitnessRate'] = (100*accountible['Ind11FitnessRate']/.75).where(
                                   accountible['Ind11ParticipationRate'] >= .9,
                                   accountible['Ind11ParticipationRate'].apply(lambda x: fitness_mult(x)))

In [22]:
keepcols = ['District', 'School Name', 'School Year', 'Year', 'Low Grade', 'High Grade', 'Category']

accountible_all = pd.melt(accountible, id_vars=keepcols, value_vars=_indicators, var_name='Indicator', value_name='Score')
accountible_all['Needs'] = accountible_all['Indicator'].map(needs_map)
accountible_all['Indicator'] = accountible_all['Indicator'].map(indicator_map)

# make numerical
accountible_all['Score'] = accountible_all['Score'].where(accountible_all['Score'] != ".", np.nan).astype('float')
accountible_all['Score'] = accountible_all['Score'].where(accountible_all['Score'] > 1, 100*accountible_all['Score'])

# get multiplier
accountible_all['Divisor'] = accountible_all['Indicator'].apply(lambda x: get_divisor(x))
accountible_all['Index']  = accountible_all['Score'] / accountible_all['Divisor']

# different calc for absenteeism
accountible_all['Index']  = accountible_all['Index'].where(~accountible_all['Indicator'].str.startswith('Chronic'),
                                                           (30 - accountible_all['Score'])/.25)


accountible_all

,District,School Name,School Year,Year,Low Grade,High Grade,Category,Indicator,Score,Needs,Divisor,Index
0,Branford School District,District,2024,2024-25,-1,-1,District,ELA Performance Index,68.669769,All,0.75,91.559692
1,Killingly School District,District,2024,2024-25,-1,-1,District,ELA Performance Index,62.716166,All,0.75,83.621555
2,Lebanon School District,District,2024,2024-25,-1,-1,District,ELA Performance Index,70.966339,All,0.75,94.621785
3,Hebron School District,District,2024,2024-25,-1,-1,District,ELA Performance Index,76.500317,All,0.75,102.000423
4,Kent School District,District,2024,2024-25,-1,-1,District,ELA Performance Index,79.092475,All,0.75,105.456633
...,...,...,...,...,...,...,...,...,...,...,...,...
406329,Connecticut Technical Education and Career System,Windham Technical High School,2014,2014-15,9,12,School,Arts Access,18.285700,All,0.60,30.476167
406330,Norwich Free Academy District,Norwich Free Academy,2014,2014-15,9,12,School,Arts Access,48.675300,All,0.60,81.125500
406331,The Gilbert School District,The Gilbert School,2014,2014-15,7,12,School,Arts Access,24.649900,All,0.60,41.083167
406332,The Woodstock Academy District,The Woodstock Academy,2014,2014-15,9,12,School,Arts Access,57.581800,All,0.60,95.969667


In [23]:
accountible_all.query('Category == "District"').groupby('Indicator')['Index'].mean()

Indicator
4-year Graduation                           95.139372
6-year Graduation                           92.753425
Arts Access                                 85.973097
Chronic Absenteeism                         59.187922
ELA Academic Growth                         60.120089
ELA Performance Index                       89.067067
Math Academic Growth                        63.425605
Math Performance Index                      82.822742
Participation Rate ELA                      97.616699
Participation Rate Math                     97.400963
Participation Rate Science                  97.604568
Physical Fitness                            56.333208
Physical Fitness Participation Rate         84.644862
Postsecondary Entrance Rate                 95.455528
Preparation for CCR (Courses)              103.618424
Preparation for CCR (Exams)                 61.984594
Progress Toward English (Literacy)          87.823593
Progress Toward English (Oral)              82.415078
Science Performanc

#### The school-level accountability file
This file contains results at both the school and district level. It has useful columns that may be used to identify the type of school, for example charter schools, that may be used to filter results.

Not otherwise used because benchmarking on a district level.

In [24]:
acctfiles = '/Users/joel/Desktop/Granby Benchmarking/education/nextgenacct*.csv'
flist = sorted(glob.glob(acctfiles))

dfs = []
for f in flist:
    dd = pd.read_csv(f, header=0, nrows=1, encoding='cp1252')
    yr = dd.columns[0][-8:]
    _df = pd.read_csv(f, skiprows=2, header=0, encoding='cp1252')
    _df['Year'] = yr.strip()
    dfs.append(_df)

acct = pd.concat(dfs)

acct['District Code'] = acct['ReportingDistrictCode'].apply(lambda x: re.search(r'.*"(\d+)"',x).group(1))
acct['School Code'] = acct['SchoolCode'].apply(lambda x: re.search(r'.*"(\d+)"',x).group(1))
acct['Code'] = acct['District Code'].str.slice(0,3).astype('int')
acct['LowGrade'] = acct['SchoolLowGrade'].str.replace('PK', '0').str.replace('K', '0').str.replace('District', '-1').astype('int')
acct['HighGrade'] = acct['SchoolHighGrade'].str.replace('PK', '0').str.replace('K', '0').str.replace('District', '-1').astype('int')

#drop district code - inconsistent
# keep Code for DRG
acct.drop(columns=['District Code'], inplace=True)

acct.drop(columns=['ReportingDistrictCode', 'SchoolCode'], inplace=True)
acct.rename(columns={'RptngDistrictName': 'District'}, inplace=True)
del dfs

In [25]:
acct.head()

,FallOfYear,District,SchoolName,SchoolOrgType,SchoolLowGrade,SchoolHighGrade,SchoolTitleIType,Category,TotalPoints,TotalPossiblePoints,...,Math_ALL_YN,ELA_HN_YN,Math_HN_YN,Improve,schoolyear,Year,School Code,Code,LowGrade,HighGrade
0,2024.0,Branford School District,District,District,District,District,NaN,DistrictTot,1095.135326,1450.0,...,.,.,.,.,2024-25,2024-25,0000000,14,-1,-1
1,2024.0,Killingly School District,District,District,District,District,NaN,DistrictTot,1017.061908,1450.0,...,.,.,.,.,2024-25,2024-25,0000000,69,-1,-1
2,2024.0,Lebanon School District,District,District,District,District,NaN,DistrictTot,1036.719991,1350.0,...,.,.,.,.,2024-25,2024-25,0000000,71,-1,-1
3,2024.0,Hebron School District,District,District,District,District,NaN,DistrictTot,640.407793,800.0,...,.,.,.,.,2024-25,2024-25,0000000,67,-1,-1
4,2024.0,Kent School District,District,District,District,District,NaN,DistrictTot,604.382338,800.0,...,.,.,.,.,2024-25,2024-25,0000000,68,-1,-1


In [26]:
acct.Category.value_counts()

Category
SchoolTot      9169
DistrictTot    1804
StateTot          9
Name: count, dtype: int64

In [27]:
acct.schoolyear.value_counts().sort_index()

schoolyear
2014-15    1246
2015-16    1246
2016-17    1241
2017-18    1230
2018-19    1222
2021-22    1202
2022-23    1202
2023-24    1199
2024-25    1194
Name: count, dtype: int64

## School Classifications
Useful for filtering to relevant institutions such as public school districts, K-12, and eliminating charter schools that have different cost structures and obligations.
Uses Accountability Index, so must follow, and used for expenditures so must precede.

#### Original District Reference Groups (DRG)

In [28]:
# this is a good place to read in the "District Reference Group"
# see separate workbook
drg = pd.read_csv('/Users/joel/Desktop/Granby Benchmarking/education/DRG.csv')

#### Type of school

Separate charter from the public school districts. Created by cross-referencing school types from the Accountability Index data and the 'organization type' from EdSight attrition data

In [29]:
org_types = acct[acct.SchoolOrgType != 'District'][['District', 'Code', 'SchoolOrgType']]\
            .value_counts().reset_index()[['District', 'Code', 'SchoolOrgType']].value_counts().reset_index().sort_values(by='Code')

grade_types = acct[acct.SchoolOrgType != 'District'].groupby(['District', 'Code' ])\
              .agg({'LowGrade':'min', 'HighGrade':'max'}).reset_index().sort_values(by='Code')

org_types = org_types.merge(grade_types, on=['District', 'Code'], how='left')\
                     .merge(drg, on='Code', how='left').drop(columns=['count'])


In [30]:
org_types[~org_types.DRG.isnull()]['SchoolOrgType'].value_counts()

SchoolOrgType
Public Schools                                149
Regional Schools                               17
Endowed and Incorporated Academies Schools      3
Name: count, dtype: int64

In [31]:
school_types = attrition.groupby(['Code', 'District'])[['Low Grade', 'High Grade', 'Organization Type']].max().reset_index()\
                        .merge(drg, on='Code', how='left')

In [32]:
school_types['Organization Type'].value_counts()

Organization Type
Public School Districts                         149
Public Charter School Districts                  22
Regional School Districts                        17
Endowed and Incorporated Academies Districts      3
Name: count, dtype: int64

In [33]:
school_types['DRG'].value_counts()

DRG
E    35
C    30
D    24
B    21
G    17
F    17
H     9
A     9
I     7
Name: count, dtype: int64

### Updated Reference Groups
See separate notebook for data, methodology, and results.

In [34]:
jsg = pd.read_parquet('/Users/joel/Desktop/Granby Benchmarking/education/dataframes/new_district_groups.parquet')

### Add school type to accountability

In [35]:
school_types = school_types.merge(jsg, on='District', how='left')

In [36]:
accountability = accountability.merge(school_types, how='left', on='District')

In [37]:
school_types[school_types['District'].str.startswith('Granby')]

,Code,District,Low Grade,High Grade,Organization Type,DRG,peer,augmented,aspirational,peerplus,...,Home Values,% w/Bachelors+,% Management,% Single Parent,% Non-English,% SNAP,School Enrollment,% HH w/Children,% w/Broadband,% Age 65+
49,56,Granby School District,PreK,12,Public School Districts,B,2.0,2.0,2.0,2.0,...,334900.0,56.4,52.9,2.4,8.7,3.1,2341.0,30.8,94.3,33.7


In [38]:
school_types[school_types['DRG'] == 'B'].head(8)

,Code,District,Low Grade,High Grade,Organization Type,DRG,peer,augmented,aspirational,peerplus,...,Home Values,% w/Bachelors+,% Management,% Single Parent,% Non-English,% SNAP,School Enrollment,% HH w/Children,% w/Broadband,% Age 65+
3,4,Avon School District,PreK,12,Public School Districts,B,0.0,1.0,1.0,0.0,...,434000.0,70.5,72.0,3.300000,20.8,2.3,4839.0,32.8,91.8,35.4
14,18,Brookfield School District,PreK,12,Public School Districts,B,0.0,0.0,0.0,1.0,...,422000.0,51.0,49.2,4.505316,15.6,2.2,4042.0,34.2,94.2,37.9
20,25,Cheshire School District,PreK,12,Public School Districts,B,0.0,1.0,0.0,1.0,...,400200.0,62.1,64.4,4.202752,14.2,3.3,7478.0,32.7,91.7,34.6
45,51,Fairfield School District,PreK,12,Public School Districts,B,0.0,1.0,1.0,0.0,...,704100.0,70.5,60.6,5.002536,14.6,2.0,20733.0,36.2,95.6,31.0
46,52,Farmington School District,PreK,12,Public School Districts,B,0.0,1.0,0.0,1.0,...,375700.0,65.1,63.8,2.605078,25.4,4.4,6433.0,27.3,94.1,34.6
48,54,Glastonbury School District,PreK,12,Public School Districts,B,0.0,1.0,1.0,0.0,...,397200.0,67.2,68.6,2.400353,17.2,3.5,8488.0,28.2,93.0,35.1
49,56,Granby School District,PreK,12,Public School Districts,B,2.0,2.0,2.0,2.0,...,334900.0,56.4,52.9,2.400000,8.7,3.1,2341.0,30.8,94.3,33.7
50,57,Greenwich School District,PreK,12,Public School Districts,B,0.0,0.0,0.0,0.0,...,1586900.0,71.1,60.9,5.604324,25.2,3.2,17726.0,37.6,95.7,32.2


In [39]:
school_types[school_types.JSG_ORIG == 6].head(10)

,Code,District,Low Grade,High Grade,Organization Type,DRG,peer,augmented,aspirational,peerplus,...,Home Values,% w/Bachelors+,% Management,% Single Parent,% Non-English,% SNAP,School Enrollment,% HH w/Children,% w/Broadband,% Age 65+
0,1,Andover School District,PreK,6,Public School Districts,C,1.0,0.0,0.0,1.0,...,366700.0,49.0,50.3,5.212953,5.1,3.9,694.0,24.8,96.5,36.8
4,5,Barkhamsted School District,PreK,6,Public School Districts,C,1.0,0.0,0.0,1.0,...,325100.0,49.8,52.8,5.705772,6.9,2.1,839.0,32.3,92.0,41.3
18,23,Canton School District,PreK,12,Public School Districts,C,1.0,0.0,0.0,1.0,...,388100.0,56.6,54.3,2.300000,5.9,2.6,2529.0,27.2,94.0,38.2
22,27,Clinton School District,PreK,12,Public School Districts,D,1.0,0.0,0.0,1.0,...,333200.0,43.0,43.9,2.623573,9.2,5.7,2733.0,26.4,93.4,38.1
23,28,Colchester School District,PreK,12,Public School Districts,D,1.0,0.0,0.0,1.0,...,315500.0,45.8,57.6,4.003777,5.5,4.9,3461.0,31.4,92.3,29.8
27,32,Coventry School District,PreK,12,Public School Districts,E,1.0,0.0,0.0,1.0,...,287800.0,45.9,49.9,5.405273,5.8,5.4,2624.0,27.2,94.4,31.4
28,33,Cromwell School District,PreK,12,Public School Districts,D,1.0,0.0,0.0,1.0,...,288600.0,44.6,50.9,5.603209,12.9,5.1,3065.0,28.1,87.4,33.0
35,41,East Haddam School District,PreK,12,Public School Districts,E,1.0,0.0,0.0,1.0,...,342000.0,43.3,51.0,5.200000,7.0,2.8,2106.0,24.7,94.1,40.7
36,42,East Hampton School District,PreK,12,Public School Districts,D,1.0,0.0,0.0,1.0,...,313500.0,36.2,46.0,3.809691,8.0,5.1,2914.0,34.2,93.9,35.2
49,56,Granby School District,PreK,12,Public School Districts,B,2.0,2.0,2.0,2.0,...,334900.0,56.4,52.9,2.400000,8.7,3.1,2341.0,30.8,94.3,33.7


## Expenditures by Object 2017-Present
High level (salaries, benefits, purchased services). Not assigned to teaching / non-teaching.

Be aware that tuition is not applied to all students - applies to a different "basis."  

Pupil Basis: 
- 1 = Enrollment plus outplaced pupils,
- 2 = Enrollment in district schools,
- 4 = Outplaced pupils

*Tuition expenditures are included with Other expenditures when fewer than six students are outplaced.

### Subset of Expenditures Wanted
Used as an estimate of overhead costs. Salaries are not subdivided by instructional, admin, custodial.

**Purchased Services** – Services rendered by organizations or
personnel not on the payroll of the school district. Although
a product may or may not result from the transaction, the
primary reason for the purchase is the service provided.  
**Supplies** – Items that are consumed, are worn out, or have
deteriorated through use or items that lose their identity
through fabrication or incorporation into different or more
complex units or substances. Includes, for example, 
instructional supplies such as textbooks, office supplies,
electricity, and fuel.  
**Property** – Purchases of equipment, furniture, equipment,
and for minor school construction projects such as for roof
replacement, energy conservation, or updates to comply
with building codes.  
**Other** – Expenditures not included in the other objects.  
**Tuition** – Line shown if greater than 5 applicable pupils.
Reimbursements to other educational entities for
instructional services to students residing within the legal
boundaries of the paying school district.


**Compare to Budget Book**  
**Largest Benefit Increase + \$346K**: Board's contribution to the Granby Health Benefit Fund for employee medical/dental insurance. The budget includes a slight premium increase, as well as census changes. Additionally, **commencing in FY27,** additional OPEB contributions are being paid by the Board of Education to meet the Actuarily Determined Contribution (ADC).  

**Purchased Services**:  
**Educational Services \$595K**: Includes funds for services including copiers, substitutes, curriculum development activities, **school resource officer benefits**, purchased instructional services for virtual classes, etc.   
**Support Services \$540K**: Includes funds for health service and physician fee contracted services. Contracted nursing services will increase 5% in FY27.  
**Support Services \$262K**: Includes funds for special education support services for evaluation services required by law. This budget also supports the Alternative Learning Center at the middle school/high school and training for teachers to deliver specialized reading instruction. The increase reflects **higher School Resource Officer benefits**, increased subsitute service rates, and hazardous waste disposal costs for science laboratories.  
**P/L Insurance \$120K**: Includes funds for property, auto, legal liability and cyber insurance.  
**Legal Fees \$69K**: Includes funds for legal fees associated with legal compliance, employee relations and grievances, internal investigations, risk management and defense, complex matters surrounding education, as well as collective bargaining, complex special education matters and due process hearings.

In [40]:
fundingfiles = '/Users/joel/Desktop/Granby Benchmarking/education/EFSDistricLevelbyObjectExport*'
flist = sorted(glob.glob(fundingfiles))

# remove special chars to create numbers
dfs = []
for f in flist:
    dd = pd.read_csv(f, header=0, nrows=1, encoding='cp1252')
    yr = dd.columns[0][-8:]
    _df = pd.read_csv(f, skiprows=10, header=0, encoding='cp1252')
    _df['Year'] = yr.strip()
    _df['Expenditures'] = _df['Expenditures'].str.replace("$", "").str.replace(",","").astype('float')
    _df['PPE'] = _df['PPE'].str.replace("$", "").str.replace(",","").astype('float')
    dfs.append(_df)
object_exp = pd.concat(dfs)

# clean up town & district names
object_exp['Town'] = object_exp['District'].apply(lambda x: get_town(x))
object_exp['Type'] = object_exp['District'].apply(lambda x: get_type(x))
object_exp['District Code'] = object_exp['District Code'].apply(lambda x: re.search(r'.*"(\d+)"',x).group(1))
object_exp['Code'] = object_exp['District Code'].str.slice(0,3).astype('int')
object_exp['School Year'] = object_exp['Year'].str.slice(0,4)

#drop district code - inconsistent
object_exp.drop(columns=['District Code'], inplace=True)


In [41]:
object_exp.groupby('Year')['Pupils'].sum()

Year
2017-18    3768609
2018-19    3738563
2019-20    3717971
2020-21    3616439
2021-22    3620870
2022-23    3619066
2023-24    3612964
2024-25    3581247
Name: Pupils, dtype: int64

#### Inflation adjustment
Personal Consumption Expenditures used for employee benefits & salaries to account for cost-of-living; Implicit Price Deflator used for other expenditures.

In [42]:
# inflation adjustment
object_exp = object_exp.merge(ipd, on='School Year', how='left').merge(pce, on='School Year', how='left')
object_exp['deflator'] = object_exp['PCE'].where(object_exp['Object'].isin(['Employee Benefits', 'Salaries']), object_exp['IPD'])
object_exp['Real Expenditures'] = np.round(object_exp['Expenditures'] / object_exp['deflator'], 2)
object_exp['Real PPE'] = np.round(object_exp['PPE'] / object_exp['deflator'], 2)

del dfs

In [43]:
object_exp.head(3)

,District,Object,Expenditures,Pupils,Pupil Basis,PPE,Year,Town,Type,Code,School Year,IPD,PCE,deflator,Real Expenditures,Real PPE
0,Achievement First Bridgeport Academy District,Salaries,8111945.0,1098,2,7388.0,2023-24,Academy,Academy,285,2023,1.28542,1.2166,1.21660,6667717.41,6072.66
1,Achievement First Bridgeport Academy District,Employee Benefits,1427869.0,1098,2,1300.0,2023-24,Academy,Academy,285,2023,1.28542,1.2166,1.21660,1173655.27,1068.55
2,Achievement First Bridgeport Academy District,Purchased Services,5267847.0,1098,2,4798.0,2023-24,Academy,Academy,285,2023,1.28542,1.2166,1.28542,4098152.35,3732.63


#### who is missing pupils?

In [44]:
object_exp.shape[0], object_exp[object_exp['Pupils'] > 1].shape[0]

(12400, 12161)

In [45]:
# most line items missing Pupil count are not Town or District (Charter, Academy)
object_exp[~(object_exp['Pupils'] > 1)]['Type'].value_counts()

Type
Charter    96
Other      76
Academy    62
Town        5
Name: count, dtype: int64

In [46]:
# And only apply to tuition
object_exp[~(object_exp['Pupils'] > 1)]['Object'].value_counts()

Object
Tuition    239
Name: count, dtype: int64

In [47]:
pd.DataFrame(object_exp.groupby(['Object', 'Pupil Basis'])[['PPE', 'Real PPE']].mean().sort_values(by='PPE')).style.format(precision=0, thousands=",", decimal=".")

,,PPE,Real PPE
Object,Pupil Basis,,
Other,2,125,108
Property,2,379,325
Other – Includes Tuition,1,631,526
Supplies,2,899,773
Purchased Services,2,"3,264","2,807"
Employee Benefits,2,"3,418","3,068"
Salaries,2,"12,933","11,605"
Total,1,"20,946","18,038"
Tuition,4,"32,669","28,020"


### Prep to insert into functional expenditures
This is for convenience, and not a problem so long as one does not sum across function becuase objects are another breakdown

In [48]:
# Year is e.g., '2017-18' like functional expenditures
object_values = ['Real Expenditures', 'Real PPE', 'Expenditures', 'PPE', 'Pupils']

# figure out what we will add to functional expenditures & what 'subobjects' to combine
# exclude total, don't need to duplicate
objects = ['Salaries', 'Employee Benefits', 'Tuition', 'Other – Includes Tuition',
           'Purchased Services', 'Supplies', 'Property', 'Other']

services = ['Purchased Services', 'Supplies', 'Property', 'Other']

# convert 'object' to 'function' - and combine services into one type, tuition into another
object_map = {x:x for x in objects} | {x: 'Services And Supplies' for x in services} | {x : 'Tuitition' for x in objects if x.find('Tuit') >= 0}

# drop "Total" - keep only objects
object_exp = object_exp.query('Object.isin(@objects)').copy()

# map Object to Function
object_exp['Function'] = object_exp['Object'].map(object_map)

# remember, town is a category, not the actual town name
object_func_exp = object_exp.groupby(['District', 'Town', 'Type', 'Code', 'School Year', 'Year', 'Function'])\
                            .agg({x: 'sum' for x in object_values} | {'Pupil Basis': 'max'}).reset_index(drop=False)

# NOW add district-levl data (functional rollup added after concat w/function)
object_exp = object_exp.merge(school_types, how='left', on=['District', 'Code'])

## Expenditures By Function
Functional expenditures are more useful because they describe "what's going on" at the school.

From https://edsight.ct.gov/relatedreports/ReportNotes_PPR.pdf


Instructional Staff and Services
- Expenditures for salaries and employee benefits, purchased services and other program
expenditures.

Instructional Supplies and Equipment
- Expenditures for expendable instructional materials, textbooks, workbooks, and textbook
binding and repairs. Expenditures for equipment, such as computers and VCRs, that is used
for direct instructional use, not for administrative or non-instructional purposes.

Improvement of Instruction and Educational Media Services
- Expenditures for activities primarily for assisting instructional staff in planning, developing, 
and evaluating the process of providing learning experiences for students. These activities
include curriculum development, techniques of instruction, staff training, etc. Also,
expenditures for activities concerned with the use of all teaching and learning resources,
including hardware, and content materials. Includes supervision of educational media
services, school library services, audiovisual services, educational television services,
computer-assisted instruction services, etc.

Student Support Services
- Expenditures for activities designed to improve students’ well-being and educational
experience. These activities include social work, guidance, health, psychological and speech
and hearing services.

Administration and Support Services
- Expenditures for the general administration of the board of education, the superintendent,
and the principals’ offices. Expenditures for other activities, including those associated with
fiscal and business support services, research, planning, evaluation, information, and data
processing, etc.

Plant Operation and Maintenance
- Expenditures for the operation and maintenance of school buildings and property, including
expenditures for utilities and heat.

Transportation
- The reimbursable and non-reimbursable (e.g., expenditures for field trips) transportation
expenditures, including vehicles, supplies, salaries, and fringe benefits. To obtain the
expenditures per pupil, the expenditures were divided by the number of resident students
enrolled in district schools or placed in schools outside the district.

Costs for Students Tuitioned Out
- Expenditures for tuition for resident students educated outside the school district. No per
pupil amount is calculated. Expenditures for secondary students from towns with
elementary school districts were not included in the elementary school districts’
expenditures.

Other
- Expenditures made from local appropriations that support student activities, excluding
funds raised by students and booster groups. These local appropriations support salaries of
coaches, activity supervisors, support services personnel covering student events, band
instruments, uniforms, and facility rentals. Also, expenditures funded by local tax dollars for
providing food to pupils and staff.

Total
- Total of all expenditures above. Expenditures for secondary students from towns with
elementary school districts were not included in the elementary school districts’
expenditures.

Basis:
- 1 = Enrollment plus outplaced pupils,
- 2 = Enrollment in district schools,
- 3 = Total pupils transported

### 2017-Present
Functional expenditures are more useful than expenditures by object for analysis. Functional expenditures separate expenses by instructional, administrative, student support, etc.

#### Pupil basis
For functional expenditures...  
1 = Enrollment plus outplaced pupils,  
2 = Enrollment in district schools,  
3 = Total pupils transported

In [49]:
fundingfiles = '/Users/joel/Desktop/Granby Benchmarking/education/EFSDistricLevelbyFunction*'
flist = sorted(glob.glob(fundingfiles))
dfs = []
for f in flist:
    dd = pd.read_csv(f, header=0, nrows=1, encoding='cp1252')
    yr = dd.columns[0][-8:]
    _df = pd.read_csv(f, skiprows=9, header=0, encoding='cp1252')
    _df['Year'] = yr.strip()
    _df['Expenditures'] = _df['Expenditures'].str.replace("$", "").str.replace(",","").astype('float')
    _df['PPE'] = _df['PPE'].str.replace("$", "").str.replace(",","").astype('float')
    dfs.append(_df)

function_exp = pd.concat(dfs)
function_exp['Town'] = function_exp['District'].apply(lambda x: get_town(x))
function_exp['Type'] = function_exp['District'].apply(lambda x: get_type(x))
function_exp['District Code'] = function_exp['District Code'].apply(lambda x: re.search(r'.*"(\d+)?"',x).group(1))
function_exp['District Code'] = function_exp['District Code'].str.slice(0,3)
function_exp['Code'] = function_exp['District Code'].str.slice(0,3).astype('int')

#drop district code - inconsistent
function_exp.drop(columns=['District Code'], inplace=True)

# add objects to the function list 
function_exp = pd.concat([function_exp, object_func_exp], ignore_index=True)

del dfs

In [50]:
function_exp.head(3)

,District,Function,Expenditures,Pupils,Pupil Basis,PPE,Year,Town,Type,Code,School Year,Real Expenditures,Real PPE
0,Achievement First Bridgeport Academy District,Instruction,9190429.0,1098.0,1.0,8370.0,2023-24,Academy,Academy,285,NaN,NaN,NaN
1,Achievement First Bridgeport Academy District,Support services - students,1180418.0,1098.0,2.0,1075.0,2023-24,Academy,Academy,285,NaN,NaN,NaN
2,Achievement First Bridgeport Academy District,Support services - instruction,270494.0,1098.0,2.0,246.0,2023-24,Academy,Academy,285,NaN,NaN,NaN


In [51]:
function_exp.Year.value_counts().sort_index()

Year
2017-18    3152
2018-19    3136
2019-20    3120
2020-21    3104
2021-22    3120
2022-23    3120
2023-24    3120
2024-25    3104
Name: count, dtype: int64

In [52]:
pd.DataFrame(function_exp.groupby(['Function', 'Pupil Basis'])['PPE'].mean().sort_values()).style.format(precision=0, thousands=",", decimal=".")

,,PPE
Function,Pupil Basis,
Food services,2.000000,26
Minor school construction,2.000000,87
Enterprise operations,2.000000,169
Tuitition,1.000000,631
Central and other support services,2.000000,754
Support services - general administration,2.000000,757
Support services - instruction,2.000000,812
Support services - school based administration,2.000000,"1,372"
Student transportation services,3.000000,"1,566"


#### who is missing pupils?

In [53]:
function_exp.shape[0], function_exp[function_exp['Pupils'] > 1].shape[0]

(24976, 23517)

In [54]:
function_exp[~(function_exp['Pupils'] > 1)]['Type'].value_counts()

Type
Town        898
Charter     202
Other       144
Academy     113
Regional    102
Name: count, dtype: int64

In [55]:
# Town & Regional Pupil Basis mostly missing for minor school construction
function_exp[~(function_exp['Pupils'] > 1) & (function_exp['Type'].isin(('Town', 'Regional')))]['Function'].value_counts()

Function
Minor school construction    995
Tuitition                      5
Name: count, dtype: int64

In [56]:
function_exp = function_exp.merge(enrollment[enrollment['Student Group'] == 'Total'].rename(columns={'School Year': 'Year'})[['District', 'Year', 'Student Count']], 
                                   how='left', on=['District', 'Year'])

function_exp['Pupils'] = function_exp['Pupils'].fillna(function_exp['Student Count'])

In [57]:
# Town & Regional Pupil Basis no longer missing for minor school construction
function_exp[~(function_exp['Pupils'] > 1) & (function_exp['Type'].isin(('Town', 'Regional')))]['Function'].value_counts()

Function
Tuitition    5
Name: count, dtype: int64

In [58]:
function_exp[(function_exp.District.str.startswith('Gran')) & (function_exp.Function.str.startswith('Student tr'))]\
[['District', 'Year', 'Expenditures', 'Pupils']].sort_values(by='Year')

,District,Year,Expenditures,Pupils
14827,Granby School District,2017-18,1460446.0,1906.0
12475,Granby School District,2018-19,1596955.0,1746.0
10135,Granby School District,2019-20,1406542.0,1670.0
7807,Granby School District,2020-21,1559556.0,1615.0
5479,Granby School District,2021-22,1430656.0,1639.0
3139,Granby School District,2022-23,2319397.0,1626.0
799,Granby School District,2023-24,2325183.0,1497.0
17203,Granby School District,2024-25,2192076.0,1538.0


### Make district-to-code mapping file
The district "code" is a way to match districts without relying on strings. Useful for older data.

In [59]:
district_code_map = function_exp[['District', 'Code']].drop_duplicates()

### 2016 and earlier
There are two separate files. One is per-pupil and the other is total expenditures. They can be combined to infer the number of pupils used for the calculation.

The functional categories are slightly different and may not align with later years. There may be more detail in the early files (such as for Instructional expenses) or in the later files (administration & support services)

#### Individual files

In [60]:
# source: https://portal.ct.gov/sde/fiscal-services/connecticut-public-school-expenditures-report
# originally excel files, had to manually clean up to be able to read as dataframes
fundingfiles = '/Users/joel/Desktop/Granby Benchmarking/education/FUNPERP_excel*.csv'
flist = sorted(glob.glob(fundingfiles))
dfs = []
for f in flist:
    _df = pd.read_csv(f, encoding='cp1252')
    dfs.append(_df)

early_pp = pd.concat(dfs)

early_pp = early_pp.merge(district_code_map, on='Code', how='left')
#early_pp['District Code'] = early_pp['Code'].astype(str).str.zfill(3)

early_pp.head(2)

,Code,Name,Instruction,Support - Instruction,School Admin,General Admin,Plant,Debt Service,Special Ed Transport,Regular Transport,Transportation,School Year,District
0,1,ANDOVER,12268,1607,1181,1330,5132,519,23693,670,767,2016-17,Andover School District
1,2,ANSONIA,9483,530,781,814,1124,1043,8279,402,895,2016-17,Ansonia School District


In [61]:
# rename columns to match 2017 and later
early_pp.rename(columns={'Instruction': 'Instruction',
             'Support - Instruction': 'Support services - instruction',
             'School Admin': 'Support services - school based administration',
             'General Admin': 'Support services - general administration',
             'Plant': 'Operation and maintenance of plant',
             'Transportation': 'Student transportation services',
             'School Year': 'Year'}, inplace=True)

# clean up district name to match other file
early_pp['Name'] = early_pp['Name'].str.strip().str.replace('"','').str.title()

# change from wide-to-long: one record per Function per District per Year (districts have a separate record for each function & year)
early_pp = pd.melt(early_pp, id_vars=['Code', 'Name', 'District', 'Year'], var_name='Function', value_name='PPE')\
             .sort_values(by=['Code', 'Name', 'District', 'Function']).drop(columns=['Name'])

# make sure that the expenditure column is numbers, not strings of numbers
early_pp['PPE'] = early_pp['PPE'].astype(str).str.replace(",","").astype('float')

# reformat town & type to match standards, for merging
early_pp['Town'] = early_pp['District'].apply(lambda x: get_town(x))
early_pp['Type'] = early_pp['District'].apply(lambda x: get_type(x))

del dfs

In [62]:
early_pp['Year'].value_counts().sort_index()

Year
2009-10    1494
2010-11    1494
2011-12    1494
2012-13    1494
2013-14    1422
2014-15    1494
2015-16    1494
2016-17    1494
Name: count, dtype: int64

In [63]:
early_pp.head(5)

,Code,District,Year,Function,PPE,Town,Type
6600,1,Andover School District,2016-17,Debt Service,519.0,Andover,Town
6878,1,Andover School District,2015-16,Debt Service,483.0,Andover,Town
7044,1,Andover School District,2014-15,Debt Service,441.0,Andover,Town
7210,1,Andover School District,2013-14,Debt Service,420.0,Andover,Town
7368,1,Andover School District,2012-13,Debt Service,423.0,Andover,Town


In [64]:
pd.DataFrame(early_pp.groupby('Function')['PPE'].mean().sort_values()).style.format(precision=0, thousands=",", decimal=".")

,PPE
Function,
Regular Transport,674
Support services - general administration,869
Student transportation services,881
Support services - school based administration,939
Debt Service,"1,286"
Support services - instruction,"1,532"
Operation and maintenance of plant,"1,674"
Instruction,"9,658"
Special Ed Transport,"12,815"


#### From trend reports
These have slightly different categories that the individual files, but also fewer years.

The trend reports separate total expenditures and per-pupil expenditures; these are merged and used to estimate the pupil count.

In [65]:
# make sure that expenditures are named the same as on other files
function_map = {'Instructional Staff and Services': 'Instruction',
                'Instructional Supplies and Equipment': 'Instruction',
                'Instruction and Educational Media Services': 'Instruction',
                'Student Support Services': 'Support services - students',
                'Total Expenditures': 'Total',
                'Transportation': 'Student transportation services',
                'Plant Operation and Maintenance': 'Operation and maintenance of plant'}

In [66]:
# sources
# https://public-edsight.ct.gov/overview/per-pupil-expenditures-by-function---district/total-annual-expenditures-by-type-2016-17-and-earlier?language=en_US

_exp = pd.read_csv('/Users/joel/Desktop/Granby Benchmarking/education/FunctionalfiscalResources.csv')

# clean text
_exp.columns = [c.strip() for c in _exp.columns]
_exp['District'] = _exp['District'].str.strip()
_exp['Function'] = _exp['Function'].str.strip()

# wide-to-long (each district has a separare record per function per year)
early_expenditures = pd.melt(_exp, id_vars=['District', 'Function'], var_name='Year', value_name='Expenditures')

In [67]:
# source
# https://public-edsight.ct.gov/overview/per-pupil-expenditures-by-function---district/per-pupil-expenditures-2016-17-and-earlier?language=en_US

_exp = pd.read_csv('/Users/joel/Desktop/Granby Benchmarking/education/FunctionalperPupilExpeditures.csv')

# clean text
_exp.columns = [c.strip() for c in _exp.columns]
_exp['District'] = _exp['District'].str.strip()
_exp['Function'] = _exp['Function'].str.strip()

# wide-to-long (each district has a separare record per function per year)
early_ppexpenditures = pd.melt(_exp, id_vars=['District', 'Function'], var_name='Year', value_name='PPE')

In [68]:
# combine total & per-person from earlier years

early_expenditures = early_expenditures.merge(early_ppexpenditures, how='left', on=['District', 'Function', 'Year'])\
                                       .merge(district_code_map, on='District', how='left')

# conform town & type to standards
early_expenditures['Town'] = early_expenditures['District'].apply(lambda x: get_town(x))
early_expenditures['Type'] = early_expenditures['District'].apply(lambda x: get_type(x))
#early_expenditures['District Code'] = early_expenditures['Code'].astype(str).str.zfill(3)

# infer number of students by simple algebra
early_expenditures['Pupils'] = np.round(early_expenditures['Expenditures'] / early_expenditures['PPE'],0)

# replace original function descriptors (strings) with functions that match later files
early_expenditures ['Early Function'] = early_expenditures ['Function']
early_expenditures ['Function'] = early_expenditures['Function'].map(function_map)
early_expenditures ['Function'] = early_expenditures['Function'].where(~early_expenditures['Function'].isnull(),early_expenditures['Early Function'])

# because later files have fewer categories for Instruction, need to combine those rows
eexp = early_expenditures.groupby(['Code', 'District', 'Town', 'Type', 'Function', 'Year']).agg({'Expenditures':'sum', 'Pupils':'mean'})\
                         .reset_index().sort_values(by=['Code', 'District', 'Year', 'Function'])

# fill in PPE for operation of plant using the function with the most pupils
inst_pupils = eexp[eexp['Function'] == 'Instruction'].groupby(['Code', 'District', 'Year'])['Pupils'].max().reset_index()\
             .rename(columns={'Pupils':'pupils_inst'})

admin_pupils = eexp[eexp['Function'].str.startswith('Administration and')].groupby(['Code', 'District', 'Year'])['Pupils'].max().reset_index()\
             .rename(columns={'Pupils':'pupils_admin'})

eexp = eexp.merge(admin_pupils, on=['Code', 'District', 'Year'], how='left')

eexp['Pupils'] = eexp['Pupils'].where(~eexp['Function'].str.startswith('Operation and maintenance'), eexp['pupils_admin'])

# final cleanup
eexp['PPE'] = round(eexp['Expenditures']/eexp['Pupils'], 2)
eexp.drop(columns=['pupils_admin'], inplace=True)

**Only Hartford, Winchester missing student counts on some early years**

In [69]:
pupcheck = inst_pupils.merge(enrollment[enrollment['Student Group'] == 'Total'].rename(columns={'School Year': 'Year'})[['District', 'Year', 'Student Count']], 
           how='left', on=['District', 'Year'])
((pupcheck['Student Count'] - pupcheck['pupils_inst'])/pupcheck['pupils_inst']).describe()

count    930.000000
mean       0.006174
std        0.006605
min       -0.017072
25%        0.001985
50%        0.004950
75%        0.008514
max        0.058084
dtype: float64

In [70]:
pupcheck[(pupcheck['Student Count'] > 0) & (pupcheck['pupils_inst'].isna())]

,Code,District,Year,pupils_inst,Student Count
282,64,Hartford School District,2014-15,NaN,21952.0
283,64,Hartford School District,2015-16,NaN,21463.0
711,162,Winchester School District,2013-14,NaN,652.0
712,162,Winchester School District,2014-15,NaN,609.0
882,280,New Beginnings Inc Family Academy District,2014-15,NaN,473.0


In [71]:
eexp.Year.value_counts().sort_index()

Year
2012-13    1520
2013-14    1520
2014-15    1520
2015-16    1520
2016-17    1520
Name: count, dtype: int64

In [72]:
eexp.head(5)

,Code,District,Town,Type,Function,Year,Expenditures,Pupils,PPE
0,1,Andover School District,Andover,Town,Administration and Support Services,2012-13,609305.0,314.000000,1940.46
1,1,Andover School District,Andover,Town,Instruction,2012-13,2791330.0,314.333333,8880.16
2,1,Andover School District,Andover,Town,Operation and maintenance of plant,2012-13,439659.0,314.000000,1400.19
3,1,Andover School District,Andover,Town,Other,2012-13,0.0,NaN,NaN
4,1,Andover School District,Andover,Town,Student transportation services,2012-13,253379.0,689.000000,367.75


#### Missing pupils
Not critical to backfill right now, because we have pupil count for important categories

In [73]:
eexp[~(eexp['Pupils'] > 1)]['Type'].value_counts()

Type
Town        871
Charter     199
Academy     131
Regional     90
Other        25
Name: count, dtype: int64

In [74]:
# mostly missing for tuitioned out / other
eexp[~(eexp['Pupils'] > 1) & eexp['Type'].isin(('Town', 'Regional'))]['Function'].value_counts()

Function
Students Tuitioned Out                 830
Other                                  107
Administration and Support Services      4
Instruction                              4
Operation and maintenance of plant       4
Student transportation services          4
Support services - students              4
Total                                    4
Name: count, dtype: int64

In [75]:
eexp[~(eexp['Pupils'] > 1) & eexp['Type'].isin(('Town', 'Regional')) & (eexp['Function'] == "Instruction")]

,Code,District,Town,Type,Function,Year,Expenditures,Pupils,PPE
2257,64,Hartford School District,Hartford,Town,Instruction,2014-15,0.0,NaN,NaN
2265,64,Hartford School District,Hartford,Town,Instruction,2015-16,0.0,NaN,NaN
5689,162,Winchester School District,Winchester,Town,Instruction,2013-14,0.0,NaN,NaN
5697,162,Winchester School District,Winchester,Town,Instruction,2014-15,0.0,NaN,NaN


In [76]:
pd.DataFrame(eexp.groupby('Function')[['Expenditures','PPE']].mean().sort_values(by='PPE')).style.format(precision=0, thousands=",", decimal=".")

,Expenditures,PPE
Function,,
Other,"521,483",273
Student transportation services,"2,469,671",880
Support services - students,"2,757,872","1,143"
Operation and maintenance of plant,"4,220,840","1,745"
Administration and Support Services,"4,663,905","2,132"
Instruction,"27,110,798","10,384"
Total,"44,185,244","16,804"
Students Tuitioned Out,"2,440,676",nan


#### Reconcile Early Expenditures
Data manipulations are successful because we are normally (interquartile range: from the 25th to 75th percentile) within a few % of other sources.

The individual excel files only report per-pupil expenditures, whereas the trend reports include total expenditures.

In [77]:
checkdf = eexp.merge(early_pp[['District', 'Function', 'Year', 'PPE']].rename(columns={'PPE': 'PPE_check'}),
                   on=['District', 'Function', 'Year'], how='left')
checkdf['percdiff'] = (checkdf['PPE'] - checkdf['PPE_check'])/checkdf['PPE']

In [78]:
checkdf.groupby('Function')['percdiff'].describe()

,count,mean,std,min,25%,50%,75%,max
Function,,,,,,,,
Administration and Support Services,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Instruction,818.0,0.053937,0.035721,-0.113107,0.028572,0.047192,0.075961,0.239164
Operation and maintenance of plant,818.0,0.005135,0.071978,-0.740333,-0.000188,0.000009,0.000246,0.627829
Other,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Student transportation services,818.0,-0.109763,0.323268,-1.415839,-0.047704,-0.020516,-0.007700,1.000000
Students Tuitioned Out,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Support services - students,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Total,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [79]:
pd.DataFrame(early_pp.groupby('Function')['PPE'].mean().sort_values()).style.format(precision=0, thousands=",", decimal=".")

,PPE
Function,
Regular Transport,674
Support services - general administration,869
Student transportation services,881
Support services - school based administration,939
Debt Service,"1,286"
Support services - instruction,"1,532"
Operation and maintenance of plant,"1,674"
Instruction,"9,658"
Special Ed Transport,"12,815"


#### Reconcile early per-pupil expenditures

In [80]:
# Get Pupil Count to infer Expenditures
early_pp = early_pp.merge(admin_pupils, on=['Code', 'District', 'Year'], how='left')\
                   .merge(inst_pupils, on=['Code', 'District', 'Year'], how='left')\
                   .merge(enrollment[enrollment['Student Group'] == 'Total'].rename(columns={'School Year': 'Year'})[['District', 'Year', 'Student Count']], 
                          how='left', on=['District', 'Year'])

In [81]:
early_pp['Pupils'] = early_pp['pupils_inst'].where(early_pp['Function'].isin(['Support services - instruction', 'Instruction']),np.nan)
early_pp['Pupils'] = early_pp['Pupils'].where(~early_pp['Function'].isin(['Student transportation services',
                                                                           'Support services - general administration',
                                                                           'Support services - school based administration',
                                                                           'Operation and maintenance of plant']), early_pp['pupils_admin'])
# all else fails use enrollment
early_pp['Pupils'] = early_pp['Pupils'].fillna(early_pp['Student Count'])

early_pp['Expenditures'] = early_pp['Pupils'] * early_pp['PPE']
#early_pp.drop(columns=['pupils_admin', 'pupils_inst'], inplace=True)

In [82]:
early_pp.head(25).tail(10)

,Code,District,Year,Function,PPE,Town,Type,pupils_admin,pupils_inst,Student Count,Pupils,Expenditures
15,1,Andover School District,2009-10,Instruction,7883.0,Andover,Town,NaN,NaN,NaN,NaN,NaN
16,1,Andover School District,2016-17,Operation and maintenance of plant,5132.0,Andover,Town,225.0,225.000000,225.0,225.0,1154700.0
17,1,Andover School District,2015-16,Operation and maintenance of plant,1831.0,Andover,Town,250.0,250.000000,250.0,250.0,457750.0
18,1,Andover School District,2014-15,Operation and maintenance of plant,1772.0,Andover,Town,275.0,275.000000,275.0,275.0,487300.0
19,1,Andover School District,2013-14,Operation and maintenance of plant,1519.0,Andover,Town,298.0,298.000000,299.0,298.0,452662.0
20,1,Andover School District,2012-13,Operation and maintenance of plant,1358.0,Andover,Town,314.0,314.333333,315.0,314.0,426412.0
21,1,Andover School District,2011-12,Operation and maintenance of plant,1358.0,Andover,Town,NaN,NaN,314.0,314.0,426412.0
22,1,Andover School District,2010-11,Operation and maintenance of plant,1360.0,Andover,Town,NaN,NaN,334.0,334.0,454240.0
23,1,Andover School District,2009-10,Operation and maintenance of plant,1179.0,Andover,Town,NaN,NaN,NaN,NaN,NaN
24,1,Andover School District,2016-17,Regular Transport,670.0,Andover,Town,225.0,225.000000,225.0,225.0,150750.0


In [83]:
# compare early total expenditures to early "per pupil"
checkdf = eexp.merge(early_pp[['District', 'Function', 'Year', 'Expenditures']].rename(columns={'Expenditures': 'check'}),
                   on=['District', 'Function', 'Year'], how='left')
checkdf['percdiff'] = (checkdf['Expenditures'] - checkdf['check'])/checkdf['Expenditures']
checkdf.groupby('Function')['percdiff'].describe()

,count,mean,std,min,25%,50%,75%,max
Function,,,,,,,,
Administration and Support Services,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Instruction,822.0,-inf,NaN,-inf,0.028281,0.046982,0.075929,0.239164
Operation and maintenance of plant,822.0,-inf,NaN,-inf,-0.000198,0.000004,0.000244,0.627829
Other,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Student transportation services,822.0,-inf,NaN,-inf,0.021301,0.066856,0.160162,1.000000
Students Tuitioned Out,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Support services - students,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Total,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [84]:
early_pp['Year'].value_counts().sort_index()

Year
2009-10    1494
2010-11    1494
2011-12    1494
2012-13    1494
2013-14    1422
2014-15    1494
2015-16    1494
2016-17    1494
Name: count, dtype: int64

In [85]:
early_pp['Function'].value_counts()

Function
Debt Service                                      1320
Instruction                                       1320
Operation and maintenance of plant                1320
Regular Transport                                 1320
Special Ed Transport                              1320
Student transportation services                   1320
Support services - general administration         1320
Support services - instruction                    1320
Support services - school based administration    1320
Name: count, dtype: int64

In [86]:
eexp['Year'].value_counts().sort_index()

Year
2012-13    1520
2013-14    1520
2014-15    1520
2015-16    1520
2016-17    1520
Name: count, dtype: int64

In [87]:
eexp['Function'].value_counts()

Function
Administration and Support Services    950
Instruction                            950
Operation and maintenance of plant     950
Other                                  950
Student transportation services        950
Students Tuitioned Out                 950
Support services - students            950
Total                                  950
Name: count, dtype: int64

### Concatenate Functional Expenditures: 2012-Present
#### Differences
The early years do not break out administration into school-based or centralized.

School year is starting year (first 4)

In [88]:
function_exp = pd.concat([function_exp,
                    early_pp[(early_pp['Function'].isin(['Support services - instruction',
                                                         'Support services - general administration',
                                                         'Support services - school based administration']))
                            & (early_pp['Year'] >= '2012-13')],
                    eexp[eexp['Function'].isin(['Administration and Support Services',
                                                'Instruction',
                                                'Operation and maintenance of plant',
                                                'Support services - students',
                                                'Student transportation services'])]
                         ]).sort_values(by=['Code', 'District', 'Year', 'Function'])

function_exp['School Year'] = function_exp['Year'].str.slice(0,4)
function_exp.shape

(32192, 16)

### Inflation Adjustment

In [89]:
function_exp = function_exp.merge(ipd, on='School Year', how='left').merge(pce, on='School Year', how='left')

# use PCE for salaries, IPD for others
function_exp['deflator'] = function_exp['IPD'].where(function_exp['Function'].isin(['Operation and maintenance of plant',
                                                                                  'Food services',
                                                                                  'Student transportation services',
                                                                                  'Enterprise operations',
                                                                                  'Minor school construction',
                                                                                 ]), function_exp['PCE'])
# "Real" Expenditures in 2017 dollars
function_exp['Real Expenditures'] = np.round(function_exp['Expenditures'] / function_exp['deflator'], 2)
function_exp['Real PPE'] = np.round(function_exp['PPE'] / function_exp['deflator'], 2)
function_exp.shape

(32192, 19)

In [90]:
# lots of historical data
function_exp['School Year'].value_counts().sort_index()

School Year
2012    1448
2013    1424
2014    1448
2015    1448
2016    1448
2017    3152
2018    3136
2019    3120
2020    3104
2021    3120
2022    3120
2023    3120
2024    3104
Name: count, dtype: int64

In [91]:
# Lot of data points for expenditures by groups
pd.DataFrame(function_exp.groupby(['Function'])['Expenditures'].count()).style.format(precision=2, thousands=",", decimal=".")

,Expenditures
Function,
Administration and Support Services,950
Central and other support services,"1,561"
Employee Benefits,"1,561"
Enterprise operations,"1,561"
Food services,"1,561"
Instruction,"2,511"
Minor school construction,393
Operation and maintenance of plant,"2,511"
Salaries,"1,561"


In [92]:
function_exp.head(390).tail(5)

,District,Function,Expenditures,Pupils,Pupil Basis,PPE,Year,Town,Type,Code,School Year,Real Expenditures,Real PPE,Student Count,pupils_admin,pupils_inst,IPD,PCE,deflator
385,Ashford School District,Student transportation services,401817.0,426.0,3.0,943.0,2017-18,Ashford,Town,3,2017,396093.45,929.57,395.0,NaN,NaN,1.01445,1.00821,1.01445
386,Ashford School District,Support services - general administration,239036.0,394.0,2.0,607.0,2017-18,Ashford,Town,3,2017,237089.50,602.06,395.0,NaN,NaN,1.01445,1.00821,1.00821
387,Ashford School District,Support services - instruction,247046.0,394.0,2.0,627.0,2017-18,Ashford,Town,3,2017,245034.27,621.89,395.0,NaN,NaN,1.01445,1.00821,1.00821
388,Ashford School District,Support services - school based administration,547710.0,394.0,2.0,1390.0,2017-18,Ashford,Town,3,2017,543249.92,1378.68,395.0,NaN,NaN,1.01445,1.00821,1.00821
389,Ashford School District,Support services - students,393088.0,394.0,2.0,998.0,2017-18,Ashford,Town,3,2017,389887.03,989.87,395.0,NaN,NaN,1.01445,1.00821,1.00821


### Testing Re-aggregation
**Dataframe `fexp` only used to develop potential logic**


In [93]:
# aggregate administration subcategories to match earlier years
# mapping 
function_agg = {'Support services - general administration': 'Administration and Support Services',
                'Support services - school based administration': 'Administration and Support Services',
                'Central and other support services': 'Administration and Support Services'}

# get school types
fexp = function_exp.merge(school_types, on='District', how='left')

# execute remapping
fexp['Late Function'] = fexp['Function']
fexp['Function'] = fexp['Function'].map(function_agg)
fexp['Function'] = fexp['Function'].where(~fexp['Function'].isnull(),fexp['Late Function'])

# re-aggregate
fexp = fexp.groupby(['District', 'Town', 'Type', 'Organization Type', 'Function', 'Year'])\
            .agg({'Expenditures':'sum', 'Real Expenditures':'sum', 'Pupils':'mean'}).reset_index()

**Infill missing Pupils**

In [94]:
# check missing
fexp[fexp['Pupils'] > 1].shape, fexp[~(fexp['Pupils'] > 1)].shape

((26080, 9), (400, 9))

In [95]:
#show where missing
fexp[~(fexp['Pupils'] > 1)]['Function'].value_counts()

Function
Tuitition                              176
Student transportation services        156
Instruction                             18
Operation and maintenance of plant      18
Support services - students             18
Administration and Support Services     14
Name: count, dtype: int64

In [96]:
# add pupils from what is available in enrollment file
fexp = fexp.merge(enrollment[enrollment['Student Group'] == 'Total'].rename(columns={'School Year': 'Year'})[['District', 'Year', 'Student Count']], 
                  how='left', on=['District', 'Year'])

fexp['Pupils'] = fexp['Pupils'].fillna(fexp['Student Count'])
# cleanup for pres
fexp['PPE'] = round(fexp['Expenditures']/fexp['Pupils'], 2)
fexp['Real PPE'] = round(fexp['Real Expenditures']/fexp['Pupils'], 2)
fexp.shape

(26480, 12)

In [97]:
fexp.columns

Index(['District', 'Town', 'Type', 'Organization Type', 'Function', 'Year',
       'Expenditures', 'Real Expenditures', 'Pupils', 'Student Count', 'PPE',
       'Real PPE'],
      dtype='object')

In [98]:
# cleanup helped a bunch
fexp[~(fexp['Pupils'] > 1)]['Function'].value_counts()

Function
Tuitition                              176
Student transportation services         55
Administration and Support Services     13
Instruction                             13
Operation and maintenance of plant      13
Support services - students             13
Name: count, dtype: int64

In [99]:
fexp[fexp.Function == 'Instruction'].head(772).tail(5)

,District,Town,Type,Organization Type,Function,Year,Expenditures,Real Expenditures,Pupils,Student Count,PPE,Real PPE
8365,Granby School District,Granby,Town,Public School Districts,Instruction,2012-13,17578146.0,18485021.14,2103.000000,2112.0,8358.60,8789.83
8366,Granby School District,Granby,Town,Public School Districts,Instruction,2013-14,17563070.0,18216306.76,2050.666667,2065.0,8564.57,8883.11
8367,Granby School District,Granby,Town,Public School Districts,Instruction,2014-15,17904026.0,18434952.64,1953.666667,1970.0,9164.32,9436.08
8368,Granby School District,Granby,Town,Public School Districts,Instruction,2015-16,17724734.0,18187422.02,1945.333333,1961.0,9111.41,9349.26
8369,Granby School District,Granby,Town,Public School Districts,Instruction,2016-17,18036511.0,18205641.41,1872.666667,1880.0,9631.46,9721.77


In [100]:
fexp.Year.value_counts().sort_index()

Year
2012-13    1106
2013-14    1098
2014-15    1106
2015-16    1106
2016-17    1106
2017-18    2632
2018-19    2632
2019-20    2632
2020-21    2618
2021-22    2618
2022-23    2618
2023-24    2618
2024-25    2590
Name: count, dtype: int64

### Cumulative Expenditure Growth By Function vs. 2017
Using earliest reported data (2017) as the base. If it is $0 then use 2018. If neither of those bases are positive, then the growth rates are set to missing.

Convenient because
- the IPD and PCE deflators use 2017 as the base year
- 2017 is a split point for early vs. late collection methods


In [101]:
# Add "base year" - 2017 or 2018 if no 2017
function_2017 = function_exp[(function_exp['Year'] == '2017-18')].rename(columns={'PPE': 'PPE_2017',
                                                                                  'Expenditures': 'Expenditures_2017',
                                                                                  'Real PPE': 'Real_PPE_2017',
                                                                                  'Real Expenditures': 'Real_Expenditures_2017',
                                                                                  'Pupils': 'Pupils_2017'})

function_2018 = function_exp[(function_exp['Year'] == '2018-19')].rename(columns={'PPE': 'PPE_2018',
                                                                                  'Expenditures': 'Expenditures_2018',
                                                                                  'Real PPE': 'Real_PPE_2018',
                                                                                  'Real Expenditures': 'Real_Expenditures_2018',
                                                                                  'Pupils': 'Pupils_2018'})

# add the basis to the time series
function_exp = function_exp.merge(function_2017[['Town', 'District', 'Function', 'Expenditures_2017', 'PPE_2017',
                                                 'Real_Expenditures_2017', 'Real_PPE_2017','Pupils_2017']],
                                  on=['Town', 'District', 'Function'], how='left')\
                           .merge(function_2018[['Town', 'District', 'Function', 'Expenditures_2018', 'PPE_2018',
                                                 'Real_Expenditures_2018', 'Real_PPE_2018','Pupils_2018']],
                                  on=['Town', 'District', 'Function'], how='left')\
                           .merge(school_types, how='left', on=['District', 'Code'])

In [102]:
# unify bases & growth rates
function_exp['Pupils Base'] = function_exp['Pupils_2017'].where(function_exp['Pupils_2017'] > 0, function_exp['Pupils_2018'])
function_exp['PPE Base'] = function_exp['PPE_2017'].where(function_exp['PPE_2017'] > 0, function_exp['PPE_2018'])
function_exp['Expenditures Base'] = function_exp['Expenditures_2017'].where(function_exp['Expenditures_2017'] > 0, function_exp['Expenditures_2018'])

function_exp['Real PPE Base'] = function_exp['Real_PPE_2017'].where(function_exp['Real_PPE_2017'] > 0, function_exp['Real_PPE_2018'])
function_exp['Real Expenditures Base'] = function_exp['Real_Expenditures_2017'].where(function_exp['Real_Expenditures_2017'] > 0, function_exp['Real_Expenditures_2018'])

function_exp['Pupil Growth'] = function_exp['Pupils'] / function_exp['Pupils Base']
function_exp['PPE Growth'] = function_exp['PPE'] / function_exp['PPE Base']
function_exp['Expenditures Growth'] = function_exp['Expenditures'] / function_exp['Expenditures Base']

function_exp['Real PPE Growth'] = function_exp['Real PPE'] / function_exp['Real PPE Base']
function_exp['Real Expenditures Growth'] = function_exp['Real Expenditures'] / function_exp['Real Expenditures Base']

# can only have growth rates if base is nonzero
function_exp['PPE Growth'] = function_exp['PPE Growth'].where(function_exp['PPE Base'] > 0, np.nan)
function_exp['Expenditures Growth'] = function_exp['Expenditures Growth'].where(function_exp['Expenditures Base'] > 0, np.nan)
function_exp['Real PPE Growth'] = function_exp['Real PPE Growth'].where(function_exp['Real PPE Base'] > 0, np.nan)
function_exp['Real Expenditures Growth'] = function_exp['Real Expenditures Growth'].where(function_exp['Real Expenditures Base'] > 0, np.nan)

function_exp.sort_values(by=['School Year', 'District'], ascending=True, inplace=True)

In [103]:
function_exp.groupby('Year')['Town'].nunique().sort_index()

Year
2012-13    153
2013-14    153
2014-15    153
2015-16    153
2016-17    153
2017-18    153
2018-19    153
2019-20    153
2020-21    153
2021-22    153
2022-23    153
2023-24    153
2024-25    152
Name: Town, dtype: int64

In [104]:
function_exp.groupby('Function')[['Town', 'Year']].nunique()

,Town,Year
Function,,
Administration and Support Services,153,5
Central and other support services,153,8
Employee Benefits,153,8
Enterprise operations,153,8
Food services,153,8
Instruction,153,13
Minor school construction,153,8
Operation and maintenance of plant,153,13
Salaries,153,8


#### Mark Granby for each of the series
This is so that we can plot granby as a separate hue on the plots. (Maybe could use a three-level hue, we'll see.)

In [105]:
# make series for identifying Granby as a different hue on the charts
function_exp['Granby'] = function_exp['Town'].where(function_exp['Town'] == 'Granby', 'Other')
function_exp['Granby EG'] = function_exp['Expenditures Growth'].where(function_exp['Town'] == 'Granby', None)
function_exp['Granby PPEG'] = function_exp['PPE Growth'].where(function_exp['Town'] == 'Granby', None)
function_exp['Granby Real EG'] = function_exp['Real Expenditures Growth'].where(function_exp['Town'] == 'Granby', None)
function_exp['Granby Real PPEG'] = function_exp['Real PPE Growth'].where(function_exp['Town'] == 'Granby', None)

In [106]:
# Granby is in all the years -- 13 years of data for Instruction, 
function_exp[~function_exp['Granby PPEG'].isna()]['Function'].value_counts().sort_index()

Function
Central and other support services                 8
Employee Benefits                                  8
Enterprise operations                              8
Food services                                      8
Instruction                                       13
Operation and maintenance of plant                13
Salaries                                           8
Services And Supplies                              8
Student transportation services                   13
Support services - general administration         13
Support services - instruction                    13
Support services - school based administration    13
Support services - students                       13
Total                                              8
Tuitition                                          8
Name: count, dtype: int64

In [107]:
# Granby is in all the years
function_exp[~function_exp['Granby PPEG'].isna()]['Year'].value_counts().sort_index()

Year
2012-13     7
2013-14     7
2014-15     7
2015-16     7
2016-17     7
2017-18    15
2018-19    15
2019-20    15
2020-21    15
2021-22    15
2022-23    15
2023-24    15
2024-25    15
Name: count, dtype: int64

In [108]:
function_exp.sort_values(by=['District', 'Function', 'Year'], ascending=True, inplace=True)
function_exp.reset_index(inplace=True)

#### Missing data by functional expenditure
**Note** that if a per-pupil expenditure is zero in 2017, it is often zero for food services and school construction. Skip those categories because they are not that helpful.

In [109]:
function_2018[function_2018['PPE_2018'] == 0].groupby('Function').count()

,District,Expenditures_2018,Pupils_2018,Pupil Basis,PPE_2018,Year,Town,Type,Code,School Year,Real_Expenditures_2018,Real_PPE_2018,Student Count,pupils_admin,pupils_inst,IPD,PCE,deflator
Function,,,,,,,,,,,,,,,,,,
Central and other support services,14,14,14,14,14,14,14,14,14,14,14,14,14,0,0,14,14,14
Employee Benefits,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,1
Enterprise operations,91,91,91,91,91,91,91,91,91,91,91,91,91,0,0,91,91,91
Food services,140,140,140,140,140,140,140,140,140,140,140,140,140,0,0,140,140,140
Instruction,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,1
Minor school construction,140,140,140,140,140,140,140,140,140,140,140,140,140,0,0,140,140,140
Operation and maintenance of plant,2,2,2,2,2,2,2,2,2,2,2,2,2,0,0,2,2,2
Salaries,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,1
Services And Supplies,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,1


In [110]:
function_2018[(function_2018['PPE_2018'] > 0) & (function_2018['Function'] == 'Food services')]['District'].unique()

array(['Ashford School District', 'Bloomfield School District',
       'Bozrah School District', 'Canterbury School District',
       'Canton School District', 'Chaplin School District',
       'Colebrook School District', 'Cromwell School District',
       'East Haddam School District', 'East Windsor School District',
       'Enfield School District', 'Glastonbury School District',
       'Granby School District', 'Griswold School District',
       'Guilford School District', 'Hampton School District',
       'Kent School District', 'Lebanon School District',
       'Lisbon School District', 'Litchfield School District',
       'Madison School District', 'Manchester School District',
       'Meriden School District', 'Middletown School District',
       'Milford School District', 'Montville School District',
       'Naugatuck School District', 'New Fairfield School District',
       'New Hartford School District', 'Newtown School District',
       'Norfolk School District', 'North Bra

## Staffing Levels

In [111]:
ed_dir = '/Users/joel/Desktop/Granby Benchmarking/education/FTES*'
flist = sorted(glob.glob(ed_dir))
dfs = []
for f in flist:
    dd = pd.read_csv(f, header=0, nrows=1)
    yr = dd.columns[0][-8:]
    df = pd.read_csv(f, skiprows=2)
    df['Year'] = yr.strip()
    dfs.append(df)

staffing = pd.concat(dfs)
staffing['Town'] = staffing['District'].apply(lambda x: get_town(x))
staffing['Type'] = staffing['District'].apply(lambda x: get_type(x))
del dfs

In [112]:
staffing.head()

,District,Assignment Category,Educator Type,FTE Count,Year,Town,Type
0,Achievement First Bridgeport Academy District,Administrators Coordinators and Department Ch...,Certified,10.0,2024-25,Academy,Academy
1,Achievement First Bridgeport Academy District,Counselors Social Workers and School Psychol...,Certified,1.0,2024-25,Academy,Academy
2,Achievement First Bridgeport Academy District,General Education - Teachers and Instructors,Certified,41.0,2024-25,Academy,Academy
3,Achievement First Bridgeport Academy District,General Education - Paraprofessional Instructi...,Non-Certified,17.6,2024-25,Academy,Academy
4,Achievement First Bridgeport Academy District,Instructional Specialists Who Support Teachers,Certified,3.0,2024-25,Academy,Academy


### Staffing expenditures by role

Combine staff roles so that they are consistent across sources & also sensible for interpretation, like the Special Education teachers + Special Education paraprofessionals are combined into one category. Teaching assistants remain separate.

Calculate staff ratios (% of staff are teachers, student/teacher ratio, etc.) and per-role salaries/benefits.

The **Expenditures By Object** files are used to count the number of students per district and calculate "Salaries + Benefits" to compare to other kinds of Expenditures.

This is basically an exercise in "long-to-wide" dataset creation. 

In [113]:
# Average type of staff by school
staffing.groupby(['Assignment Category'])['FTE Count'].mean()

Assignment Category
Administrators  Coordinators and Department Chairs - District Central Office      5.847548
Administrators  Coordinators and Department Chairs - School Level                13.279108
Counselors  Social Workers  and School Psychologists                             18.534880
General Education - Paraprofessional Instructional Assistants                    24.606560
General Education - Teachers and Instructors                                    180.984800
Instructional Specialists Who Support Teachers                                   12.703269
Library/Media - Specialists (Certified)                                           4.615271
Library/Media - Support Staff                                                     2.602080
Other Staff Providing Non-Instructional Services/Support                        129.646221
School Nurses                                                                     6.613600
Special Education - Paraprofessional Instructional Assistants         

In [114]:
# number of pupils per district per year
pupils = object_exp[(object_exp['Object'] == 'Salaries')].groupby(['Town', 'District', 'Year'])['Pupils'].sum().reset_index()

In [115]:
# missing pupil count?
pupils[pupils['Pupils'].isna()]

,Town,District,Year,Pupils


In [116]:
pupils[pupils['Pupils'] > 0].groupby('Year')['Town'].nunique().sort_index()

Year
2017-18    153
2018-19    153
2019-20    153
2020-21    153
2021-22    153
2022-23    153
2023-24    153
2024-25    152
Name: Town, dtype: int64

#### Get teaching salaries, expenditures

In [117]:
# salaries & benefits (object) by town & year, total & per-student
# note rename for clarity
salaries = object_exp[object_exp['Object'].isin(['Salaries', 'Employee Benefits'])].groupby(['Town', 'District', 'Year'])\
            [['Expenditures', 'PPE', 'Real Expenditures', 'Real PPE']].sum().reset_index()\
            .rename(columns={'Expenditures': 'Payroll', 'Real Expenditures': 'Real Payroll', 'PPE': 'Payroll Per Student', 'Real PPE': 'Real Payroll Per Student'})

In [118]:
# instructional (functional) expenditures by town & year
instructional = function_exp[function_exp['Function'] == 'Instruction'][['Town', 'District', 'Year', 'Expenditures', 'PPE', 'Real Expenditures', 'Real PPE']]\
    .rename(columns={'Expenditures': 'Instruction $', 'PPE': 'Instruction $ Per Student', 'Real Expenditures': 'Real Instruction $', 'Real PPE': 'Real Instruction $ Per Student'})

#### Count staffing by role

In [119]:
# Roll up from detailed to 'gross' employee roles

# types of non-teaching staff
counselors = staffing[staffing['Assignment Category'].str.startswith('Counselors')].groupby(['Town', 'District', 'Year'])['FTE Count'].sum().reset_index()
admin = staffing[staffing['Assignment Category'].str.startswith('Administrators')].groupby(['Town', 'District', 'Year'])['FTE Count'].sum().reset_index()
other = staffing[staffing['Assignment Category'].str.startswith('Other Staff')].groupby(['Town', 'District', 'Year'])['FTE Count'].sum().reset_index()

# teachers
teachers = staffing[(staffing['Assignment Category'].str.startswith('General Education - Teachers'))].groupby(['Town', 'District', 'Year'])['FTE Count'].sum().reset_index()
teachers_se = staffing[staffing['Assignment Category'].str.startswith('Special Education - Teachers')].groupby(['Town', 'District', 'Year'])['FTE Count'].sum().reset_index()
special_ed = staffing[staffing['Assignment Category'].str.startswith('Special Education')].groupby(['Town', 'District', 'Year'])['FTE Count'].sum().reset_index()

# non-teacher instructional (general Ed)
assistants = staffing[(staffing['Assignment Category'].str.startswith('General Education - Paraprofessional')) |
                     (staffing['Assignment Category'].str.startswith('Instructional Specialists')) ].groupby(['Town', 'District', 'Year'])['FTE Count'].sum().reset_index()

# general ed includes teachers + assistants
education = staffing[(staffing['Assignment Category'].str.startswith('General Education')) |
                     (staffing['Assignment Category'].str.startswith('Instructional Specialists')) ].groupby(['Town', 'District', 'Year'])['FTE Count'].sum().reset_index()

# all - special & general
instruction = staffing[(staffing['Assignment Category'].str.startswith('General Education')) | (staffing['Assignment Category'].str.startswith('Special Education')) |
                         (staffing['Assignment Category'].str.startswith('Instructional Specialists')) ].groupby(['Town', 'District', 'Year'])['FTE Count'].sum().reset_index()

total = staffing.groupby(['Town', 'District', 'Year'])['FTE Count'].sum().reset_index()

#### Combine

In [120]:
enroll = enrollment.pivot(columns='Student Group', index=['District', 'School Year'], values='Student Count')\
         .reset_index(drop=False).rename(columns={'Total':'Enrollment', 'School Year':'Year'})

In [121]:
# combine staffing levels with salaries, expenses, etc.
totalstaff = total.merge(teachers.rename(columns={"FTE Count": 'Teachers'}), how='left', on=['Town', 'District', 'Year'])\
             .merge(assistants.rename(columns={"FTE Count": 'Assistants'}), how='left', on=['Town', 'District', 'Year'])\
             .merge(education.rename(columns={"FTE Count": 'Educational'}), how='left', on=['Town', 'District', 'Year'])\
             .merge(instruction.rename(columns={"FTE Count": 'Instructional'}), how='left', on=['Town', 'District', 'Year'])\
             .merge(other.rename(columns={"FTE Count": 'Other'}), how='left', on=['Town', 'District', 'Year'])\
             .merge(admin.rename(columns={"FTE Count": 'Admin'}), how='left', on=['Town', 'District', 'Year'])\
             .merge(counselors.rename(columns={"FTE Count": 'Counselors'}), how='left', on=['Town', 'District', 'Year'])\
             .merge(teachers_se.rename(columns={"FTE Count": 'SpecialEd Teachers'}), how='left', on=['Town', 'District', 'Year'])\
             .merge(special_ed.rename(columns={"FTE Count": 'SpecialEd'}), how='left', on=['Town', 'District', 'Year'])\
             .merge(salaries, how='left', on=['Town', 'District', 'Year'])\
             .merge(instructional, how='left', on=['Town', 'District', 'Year'])\
             .merge(pupils, how='left', on=['Town', 'District', 'Year'])\
             .merge(retention, how='left', on=['District', 'Year'])\
             .merge(enroll, how='left', on=['District', 'Year'])\
             .merge(school_types, how='left', on=['District'])

In [122]:
# calculate shares and ratios
for staff in ['Teachers', 'Educational', 'Assistants', 'SpecialEd Teachers', 'SpecialEd', 'Instructional', 'Other', 'Admin', 'Counselors']:
    totalstaff[f"{staff} Share Of FTE"] = totalstaff[staff] / totalstaff['FTE Count']
    totalstaff[f"{staff} Per Student"] = totalstaff[staff] / totalstaff['Enrollment']
    totalstaff[f"{staff} Per 100 Student"] = 100*totalstaff[staff] / totalstaff['Enrollment']
    totalstaff[f"Student: {staff} Ratio"] = totalstaff['Enrollment'] / totalstaff[staff]

# in-group only
totalstaff['Teachers Share of General Ed'] = totalstaff['Teachers'] / totalstaff['Educational']
totalstaff['Teachers Share of Special Ed'] = totalstaff['SpecialEd Teachers'] / totalstaff['SpecialEd']

# either or / could use 'Instructional' instead of Educational + SpecialEd
totalstaff['Certified Share of Instructional'] = (totalstaff['Teachers'] + totalstaff['SpecialEd Teachers']) / totalstaff['Instructional']
totalstaff['General Ed Share of Instructional'] = totalstaff['Educational'] / totalstaff['Instructional']
totalstaff['Special Ed Share of Instructional'] = totalstaff['SpecialEd'] /  totalstaff['Instructional']

# high-need vs. not
totalstaff[f"High Needs Students Share of Enrollment"] = totalstaff['High Needs'] / totalstaff['Enrollment']
totalstaff[f"Students w/ Disabilities Share of Enrollment"] = totalstaff['Students with Disabilities'] / totalstaff['Enrollment']
totalstaff[f"SpecialEd Per 100 Student w/ Disabilities"] = 100*totalstaff['SpecialEd'] / totalstaff['Students with Disabilities']
totalstaff[f"Educational Per 100 Student w/o Disabilities"] = 100*totalstaff['Educational'] / totalstaff['Students without Disabilities']
totalstaff[f"Teachers Per 100 Student w/o Disabilities"] = 100*totalstaff['Teachers'] / totalstaff['Students without Disabilities']

totalstaff[f"Students w/ Disabilities: Staff Ratio"] = totalstaff['Students with Disabilities'] / totalstaff['SpecialEd']
totalstaff[f"Students w/ Disabilities: Teacher Ratio"] = totalstaff['Students with Disabilities'] / totalstaff['SpecialEd Teachers']
totalstaff[f"Students w/o Disabilities: Teacher Ratio"] = totalstaff['Students without Disabilities'] / totalstaff['Teachers']
totalstaff[f"Students w/o Disabilities: Educational Ratio"] = totalstaff['Students without Disabilities'] / totalstaff['Educational']

In [123]:
# most districts match pupil from object expenses to enrollment data
((totalstaff['Enrollment'] - totalstaff['Pupils'])/totalstaff['Pupils']).describe()

count    1561.000000
mean        0.003974
std         0.018505
min        -0.217166
25%         0.000780
50%         0.005060
75%         0.008008
max         0.049242
dtype: float64

In [124]:
# categorize district type and separate Granby for plotting
totalstaff['Type'] = totalstaff['District'].apply(lambda x: get_type(x))
totalstaff['Granby'] = totalstaff['Town'].where(totalstaff['Town'] == 'Granby', 'Other')

# Try to get sizing dots for plotting; doesn't work on seaborn's faceted grids
totalstaff['Size'] = 52
totalstaff['Size'] = totalstaff['Size'].where(totalstaff['Town'] == 'Granby', 6).astype('float')

In [125]:
(totalstaff.columns)

Index(['Town', 'District', 'Year', 'FTE Count', 'Teachers', 'Assistants',
       'Educational', 'Instructional', 'Other', 'Admin',
       ...
       'SpecialEd Per 100 Student w/ Disabilities',
       'Educational Per 100 Student w/o Disabilities',
       'Teachers Per 100 Student w/o Disabilities',
       'Students w/ Disabilities: Staff Ratio',
       'Students w/ Disabilities: Teacher Ratio',
       'Students w/o Disabilities: Teacher Ratio',
       'Students w/o Disabilities: Educational Ratio', 'Type', 'Granby',
       'Size'],
      dtype='object', length=116)

In [126]:
# doess it look populated?
totalstaff[[c for c in totalstaff.columns if c.startswith('Student')]].describe()

,Student Count,Students with Disabilities,Students without Disabilities,Student: Teachers Ratio,Student: Educational Ratio,Student: Assistants Ratio,Student: SpecialEd Teachers Ratio,Student: SpecialEd Ratio,Student: Instructional Ratio,Student: Other Ratio,Student: Admin Ratio,Student: Counselors Ratio,Students w/ Disabilities Share of Enrollment,Students w/ Disabilities: Staff Ratio,Students w/ Disabilities: Teacher Ratio,Students w/o Disabilities: Teacher Ratio,Students w/o Disabilities: Educational Ratio
count,1139.000000,3184.000000,3196.000000,3183.000000,3189.000000,3176.000000,3115.000000,3190.000000,3191.000000,3167.000000,3180.000000,3011.000000,3184.000000,3184.000000,3114.000000,3177.000000,3181.000000
mean,2593.024583,403.079146,2259.972778,14.245387,inf,inf,95.305367,inf,8.359023,inf,135.774690,169.395519,0.157286,inf,13.033216,12.075017,9.823023
std,3475.780636,578.990744,3000.871024,4.337168,NaN,NaN,59.828174,NaN,2.510683,NaN,47.396247,86.561237,0.085565,NaN,5.406945,3.930460,2.881912
min,41.000000,6.000000,0.000000,1.686047,1.647727,3.540373,3.000000,2.500000,1.435644,0.781784,4.500000,9.176471,0.017804,0.677201,2.739130,0.225352,0.219178
25%,445.500000,63.000000,378.000000,12.688230,9.987138,47.346578,70.400617,22.390926,7.092936,15.862291,108.746711,124.272274,0.117830,3.341939,10.166667,10.639535,8.413284
50%,1440.000000,221.000000,1239.000000,14.178700,11.704545,74.712305,85.700748,29.774994,8.286542,20.173077,136.250000,153.142857,0.144015,4.157768,12.341526,12.026618,9.969697
75%,3225.000000,492.000000,2772.250000,15.548664,13.080663,115.824364,103.416667,38.981813,9.513100,24.657316,164.203484,189.382166,0.175430,5.539342,14.600000,13.389181,11.305031
max,21264.000000,4166.000000,19008.000000,146.000000,inf,inf,780.000000,inf,41.320755,inf,558.000000,1090.000000,1.000000,inf,76.000000,138.000000,73.928571


### Academies and Charter Schools have significantly fewer Special Ed resources

In [127]:
_ts = totalstaff[(totalstaff['High Grade'] == 12) & (totalstaff['Year'] == '2024-25')].groupby('Organization Type')['High Needs Students Share of Enrollment'].describe()
fdict = {c: '{:.2%}' for c in _ts.columns}
fdict['count'] = '{:.0f}'
_ts.style.format(fdict).set_caption("High Needs Share Of Enrollment, 2024-25")

,count,mean,std,min,25%,50%,75%,max
Organization Type,,,,,,,,
Endowed and Incorporated Academies Districts,3,45.83%,28.69%,12.82%,36.39%,59.96%,62.33%,64.71%
Public Charter School Districts,11,81.36%,7.73%,72.73%,74.03%,81.48%,85.24%,97.02%
Public School Districts,103,48.48%,19.35%,13.89%,33.32%,44.67%,63.11%,92.64%
Regional School Districts,16,34.55%,9.38%,23.31%,28.88%,31.48%,38.85%,63.40%


In [128]:
_ts = totalstaff[(totalstaff['High Grade'] == 12) & (totalstaff['Year'] == '2024-25')].groupby('Organization Type')['Students w/ Disabilities Share of Enrollment'].describe()
fdict = {c: '{:.2%}' for c in _ts.columns}
fdict['count'] = '{:.0f}'
_ts.style.format(fdict).set_caption("Students w/ Disabilities Share Of Enrollment, 2024-25")

,count,mean,std,min,25%,50%,75%,max
Organization Type,,,,,,,,
Endowed and Incorporated Academies Districts,3,11.95%,2.81%,9.00%,10.63%,12.26%,13.43%,14.60%
Public Charter School Districts,11,15.80%,9.50%,8.06%,9.58%,11.58%,17.25%,37.66%
Public School Districts,103,18.21%,2.92%,11.19%,15.93%,18.04%,20.52%,25.56%
Regional School Districts,16,17.91%,3.64%,13.41%,15.07%,17.55%,19.32%,26.29%


In [129]:
_ts = totalstaff[(totalstaff['High Grade'] == 12) & (totalstaff['Year'] == '2024-25')].groupby('Organization Type')['SpecialEd Share Of FTE'].describe()
fdict = {c: '{:.2%}' for c in _ts.columns}
fdict['count'] = '{:.0f}'
_ts.style.format(fdict).set_caption("SpecialEd Share of FTE, 2024-25")

,count,mean,std,min,25%,50%,75%,max
Organization Type,,,,,,,,
Endowed and Incorporated Academies Districts,3,7.37%,6.39%,0.00%,5.35%,10.70%,11.05%,11.40%
Public Charter School Districts,11,7.85%,3.87%,2.59%,4.50%,8.46%,10.36%,13.62%
Public School Districts,103,19.93%,4.03%,9.52%,17.36%,19.56%,22.16%,32.03%
Regional School Districts,16,17.47%,5.16%,7.86%,13.64%,17.59%,20.92%,25.55%


In [130]:
_ts = totalstaff[(totalstaff['High Grade'] == 12) & (totalstaff['Year'] == '2024-25')].groupby('Organization Type')['Teachers Share Of FTE'].describe()
fdict = {c: '{:.2%}' for c in _ts.columns}
fdict['count'] = '{:.0f}'
_ts.style.format(fdict).set_caption("Teachers Share of FTE, 2024-25")

,count,mean,std,min,25%,50%,75%,max
Organization Type,,,,,,,,
Endowed and Incorporated Academies Districts,3,37.92%,6.62%,31.64%,34.46%,37.28%,41.06%,44.84%
Public Charter School Districts,11,32.95%,6.36%,21.15%,28.97%,32.04%,36.96%,42.53%
Public School Districts,103,34.97%,3.98%,23.18%,31.90%,34.79%,36.99%,47.08%
Regional School Districts,16,35.73%,3.73%,30.92%,32.52%,35.49%,38.36%,43.37%


In [131]:
_ts = totalstaff[(totalstaff['High Grade'] == 12) & (totalstaff['Year'] == '2024-25')].groupby('Organization Type')['Educational Share Of FTE'].describe()
fdict = {c: '{:.2%}' for c in _ts.columns}
fdict['count'] = '{:.0f}'
_ts.style.format(fdict).set_caption("Educational Share of FTE, 2024-25")

,count,mean,std,min,25%,50%,75%,max
Organization Type,,,,,,,,
Endowed and Incorporated Academies Districts,3,38.66%,6.38%,32.10%,35.58%,39.06%,41.95%,44.84%
Public Charter School Districts,11,46.96%,8.43%,35.05%,37.93%,50.49%,53.60%,56.11%
Public School Districts,103,42.32%,4.07%,33.40%,39.64%,42.06%,44.71%,54.77%
Regional School Districts,16,40.33%,3.39%,35.29%,37.88%,40.63%,42.01%,48.55%


In [132]:
_ts = totalstaff[(totalstaff['High Grade'] == 12) & (totalstaff['Year'] == '2024-25')].groupby('Organization Type')['Admin Share Of FTE'].describe()
fdict = {c: '{:.2%}' for c in _ts.columns}
fdict['count'] = '{:.0f}'
_ts.style.format(fdict).set_caption("Admin Share of FTE, 2024-25")

,count,mean,std,min,25%,50%,75%,max
Organization Type,,,,,,,,
Endowed and Incorporated Academies Districts,3,5.71%,0.53%,5.13%,5.48%,5.83%,6.00%,6.17%
Public Charter School Districts,11,7.87%,3.84%,2.79%,5.39%,8.46%,8.67%,15.53%
Public School Districts,103,3.64%,0.60%,2.58%,3.17%,3.71%,3.97%,5.25%
Regional School Districts,16,4.68%,1.39%,3.14%,3.72%,4.27%,5.17%,7.78%


In [133]:
totalstaff[totalstaff['Granby']=='Granby'][['Year', 'Enrollment', 'Teachers', 'Certified Share of Instructional',
                                            'Teachers Share of General Ed', 'Teachers Share of Special Ed', 'Student: Teachers Ratio', 'Student: Educational Ratio', 
                                            'Students w/ Disabilities: Teacher Ratio', 'Students w/o Disabilities: Teacher Ratio']]\
                                            .style.format({'Enrollment': '{:,.0f}', 'Teachers': '{:,.0f}'} | 
                                                          {x: '{:,.1%}' for x in ['Certified Share of Instructional', 'Teachers Share of General Ed', 'Teachers Share of Special Ed']} |
                                                         {x: '{:.1f}' for x in ['Student: Teachers Ratio', 'Student: Educational Ratio', 
                                                                        'Students w/ Disabilities: Teacher Ratio', 'Students w/o Disabilities: Teacher Ratio']})\
                                                           .hide(axis='index')

Year,Enrollment,Teachers,Certified Share of Instructional,Teachers Share of General Ed,Teachers Share of Special Ed,Student: Teachers Ratio,Student: Educational Ratio,Students w/ Disabilities: Teacher Ratio,Students w/o Disabilities: Teacher Ratio
2007-08,nan,137,67.1%,80.6%,29.0%,nan,nan,nan,nan
2008-09,nan,134,63.4%,77.4%,25.7%,nan,nan,nan,nan
2009-10,nan,135,65.3%,78.7%,27.3%,nan,nan,nan,nan
2010-11,"2,253",135,65.2%,79.0%,28.7%,16.7,13.2,9.9,15.4
2011-12,"2,171",134,66.0%,80.4%,29.1%,16.2,13.1,10.1,14.8
2012-13,"2,112",131,67.1%,78.8%,33.0%,16.1,12.7,9.7,14.7
2013-14,"2,065",129,71.1%,83.1%,36.7%,16.0,13.3,10.1,14.5
2014-15,"1,970",130,71.1%,84.0%,36.5%,15.2,12.8,9.5,13.7
2015-16,"1,961",126,69.6%,83.4%,34.9%,15.6,13.0,10.6,13.8
2016-17,"1,880",128,70.7%,82.8%,37.4%,14.7,12.2,10.9,12.9


## Save datasets

In [134]:
totalstaff.to_parquet('/Users/joel/Desktop/Granby Benchmarking/education/dataframes/totalstaff.parquet')
function_exp.to_parquet('/Users/joel/Desktop/Granby Benchmarking/education/dataframes/expenditures.parquet')
object_exp.to_parquet('/Users/joel/Desktop/Granby Benchmarking/education/dataframes/expenditures_by_object.parquet')
object_func_exp.to_parquet('/Users/joel/Desktop/Granby Benchmarking/education/dataframes/expenditures_by_object_as_func.parquet')
accountability.to_parquet('/Users/joel/Desktop/Granby Benchmarking/education/dataframes/accountability.parquet')
accountible_all.to_parquet('/Users/joel/Desktop/Granby Benchmarking/education/dataframes/accountability_schools.parquet')
school_types.to_parquet('/Users/joel/Desktop/Granby Benchmarking/education/dataframes/school_types.parquet')